In [ ]:
# Mount Google Drive and initialize Linear SVM experiment paths

from google.colab import drive
from pathlib import Path

import json
import random
import time
import hashlib
import joblib
import platform
import sys
import gc
import warnings

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime, timezone
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.exceptions import ConvergenceWarning

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

CONFIG_DIR = PROJECT_ROOT / '00_config'
PROCESSED_DIR = PROJECT_ROOT / '01_data' / 'processed'
SPLITS_DIR = PROJECT_ROOT / '01_data' / 'splits'
FEATURES_DIR = PROJECT_ROOT / '01_data' / 'features'
SRC_DIR = PROJECT_ROOT / '03_src'

CLASSICAL_DIR = PROJECT_ROOT / '02_notebooks' / '03_classical_ml'
NOTEBOOK_DIR = CLASSICAL_DIR / '03_linear_svm'

RESULTS_DIR = NOTEBOOK_DIR / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'

MODELS_DIR = (
    PROJECT_ROOT / '04_models' / 'classical' / 'linear_svm'
)

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'

assert NOTEBOOK_DIR.is_dir(), "Notebook directory not found"

for folder in [
    TABLES_DIR,
    FIGURES_DIR,
    PREDICTIONS_DIR,
    LOGS_DIR,
    RUN_CONFIGS_DIR,
    MODELS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Load the shared project configuration

with open(CONFIG_DIR / 'project_config.json', 'r') as file:
    config = json.load(file)

LABELS = config['labels']
SEED = config['seed']
FEATURE_BUDGET = config['classical_features']['feature_budget']

MODEL_NAME = 'Linear SVM'

REPRESENTATIONS = [
    'word_tfidf',
    'character_tfidf',
    'combined_tfidf'
]

DEFAULT_THRESHOLD = float(
    config['evaluation']['default_svm_margin_threshold']
)

SCORE_TYPE = 'signed_decision_margin'

random.seed(SEED)
np.random.seed(SEED)

assert DEFAULT_THRESHOLD == 0.0
assert config['classical_features']['status'] == 'verified'
assert REGISTRY_PATH.is_file()

assert all(
    representation in config['classical_features']['representations']
    for representation in REPRESENTATIONS
)

# Reuse the verified evaluation module

sys.path.insert(0, str(SRC_DIR))

from evaluation_utils import (
    apply_thresholds,
    evaluate_multilabel,
    optimize_thresholds
)

def calculate_sha256(path):

    sha256 = hashlib.sha256()

    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            sha256.update(chunk)

    return sha256.hexdigest()


# Verify consistency with previously completed experiments

registry_df = pd.read_csv(REGISTRY_PATH)

assert registry_df['run_id'].is_unique

required_previous_runs = [
    'NB_WORD_BASELINE_S42',
    'LR_WORD_BASELINE_S42',
    'LR_CHAR_BASELINE_S42',
    'LR_COMBINED_BASELINE_S42',
    'LR_COMBINED_BALANCED_S42'
]

assert set(required_previous_runs).issubset(
    set(registry_df['run_id'])
)

evaluation_hash = calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
)

feature_manifest_hash = calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
)

previous_run_configs = {}
consistency_records = []

for run_id in required_previous_runs:

    record = registry_df.loc[
        registry_df['run_id'] == run_id
    ].iloc[0]

    assert record['status'] == 'Completed'

    with open(
        PROJECT_ROOT / record['config_path'], 'r'
    ) as file:
        run_config = json.load(file)

    assert run_config['labels'] == LABELS
    assert run_config['seed'] == SEED

    assert (
        run_config['evaluation_module_sha256']
        == evaluation_hash
    )

    assert (
        run_config['feature_manifest_sha256']
        == feature_manifest_hash
    )

    assert not run_config['official_test_used']

    previous_run_configs[run_id] = run_config

    consistency_records.append({
        'Run ID': run_id,
        'Label Order Verified': True,
        'Seed Verified': True,
        'Evaluation Module Verified': True,
        'Feature Manifest Verified': True,
        'Official Test Used': False,
        'Status': 'Verified'
    })

previous_run_verification_df = pd.DataFrame(
    consistency_records
)

print("Previous experiment consistency verification")

display(previous_run_verification_df)

previous_run_verification_df.to_csv(
    TABLES_DIR / 'previous_run_consistency.csv',
    index=False
)

# Verify shared dataset fingerprints across previous runs

training_hashes = {
    run_config['training_dataset_sha256']
    for run_config in previous_run_configs.values()
}

validation_hashes = {
    run_config['validation_dataset_sha256']
    for run_config in previous_run_configs.values()
}

assert len(training_hashes) == 1
assert len(validation_hashes) == 1

EXPECTED_TRAINING_HASH = next(iter(training_hashes))
EXPECTED_VALIDATION_HASH = next(iter(validation_hashes))


# Verify evaluation support using two-label signed margins

toy_labels = ['example_a', 'example_b']

toy_targets = np.array([
    [0, 0],
    [1, 0],
    [0, 1],
    [1, 1]
], dtype=np.int8)

toy_scores = np.array([
    [-1.5, -0.6],
    [0.8, -1.2],
    [-0.4, 0.4],
    [-0.1, 0.2]
], dtype=np.float64)

assert toy_targets.shape == toy_scores.shape

toy_predictions = apply_thresholds(
    toy_scores,
    DEFAULT_THRESHOLD
)

expected_default_predictions = np.array([
    [0, 0],
    [1, 0],
    [0, 1],
    [0, 1]
], dtype=np.int8)

assert np.array_equal(
    toy_predictions,
    expected_default_predictions
)

# Evaluate the default SVM threshold

toy_default_metrics, toy_default_per_label = evaluate_multilabel(
    y_true=toy_targets,
    y_scores=toy_scores,
    labels=toy_labels,
    thresholds=DEFAULT_THRESHOLD
)

# Optimize negative and positive decision thresholds

toy_threshold_df = optimize_thresholds(
    y_true=toy_targets,
    y_scores=toy_scores,
    labels=toy_labels
)

assert toy_threshold_df['Label'].tolist() == toy_labels

toy_thresholds = toy_threshold_df[
    'Optimal Threshold'
].to_numpy(dtype=np.float64)

assert np.isfinite(toy_thresholds).all()

assert np.allclose(
    toy_thresholds,
    [-0.1, 0.2]
)

toy_metrics, toy_per_label_df = evaluate_multilabel(
    y_true=toy_targets,
    y_scores=toy_scores,
    labels=toy_labels,
    thresholds=toy_thresholds
)

assert np.isclose(toy_metrics['Macro F1'], 1.0)

assert np.array_equal(
    apply_thresholds(toy_scores, toy_thresholds),
    toy_targets
)

assert np.allclose(
    toy_per_label_df['F1'].to_numpy(),
    [1.0, 1.0]
)

# Verify that threshold optimization preserves ranking metrics

for metric in [
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Micro ROC-AUC'
]:

    assert np.isclose(
        toy_default_metrics[metric],
        toy_metrics[metric]
    )

# Save signed-margin verification results

signed_margin_verification_df = pd.DataFrame({
    'Label': toy_labels,
    'Default Threshold': DEFAULT_THRESHOLD,
    'Optimal Threshold': toy_thresholds,
    'Default F1': toy_default_per_label['F1'].to_numpy(),
    'Optimized F1': toy_per_label_df['F1'].to_numpy(),
    'Verification': 'Passed'
})

print("\nSigned margin evaluation verification")

display(signed_margin_verification_df.round(6))

signed_margin_verification_df.to_csv(
    TABLES_DIR / 'signed_margin_evaluation_verification.csv',
    index=False
)

print("Negative and positive thresholds: Supported")
print("Ranking metric consistency: Verified")


# Configure consistent visualization settings

sns.set_theme(style='whitegrid')

LABEL_COLORS = {
    'toxic': '#4C78A8',
    'severe_toxic': '#F58518',
    'obscene': '#54A24B',
    'threat': '#E45756',
    'insult': '#B279A2',
    'identity_hate': '#72B7B2'
}

REPRESENTATION_COLORS = {
    'word_tfidf': '#4C78A8',
    'character_tfidf': '#F58518',
    'combined_tfidf': '#54A24B'
}

# Record current environment versions

environment_df = pd.DataFrame([
    {'Package': 'Python', 'Version': platform.python_version()},
    {'Package': 'NumPy', 'Version': np.__version__},
    {'Package': 'Pandas', 'Version': pd.__version__},
    {'Package': 'SciPy', 'Version': scipy.__version__},
    {'Package': 'Scikit-learn', 'Version': sklearn.__version__},
    {'Package': 'Joblib', 'Version': joblib.__version__}
])

environment_df.to_csv(
    TABLES_DIR / 'environment_versions.csv',
    index=False
)

print("\nCurrent environment versions")

display(environment_df)

# Compare environment with the feature extraction notebook

feature_environment_path = (
    CLASSICAL_DIR / '00_feature_extraction'
    / 'results' / 'tables' / 'environment_versions.csv'
)

assert feature_environment_path.is_file()

feature_environment_df = pd.read_csv(
    feature_environment_path
)

environment_comparison_df = environment_df.merge(
    feature_environment_df,
    on='Package',
    suffixes=('_Current', '_Feature_Extraction')
)

environment_comparison_df['Version Match'] = (
    environment_comparison_df['Version_Current']
    == environment_comparison_df['Version_Feature_Extraction']
)

print("\nEnvironment compatibility")

display(environment_comparison_df)

environment_comparison_df.to_csv(
    TABLES_DIR / 'environment_compatibility.csv',
    index=False
)

# Summarize initialization and verification

print("\nLinear SVM initialization summary")

print("Notebook:", NOTEBOOK_DIR.name)
print("Model:", MODEL_NAME)
print("Feature representations:", REPRESENTATIONS)
print("Common feature budget:", FEATURE_BUDGET)
print("Target labels:", LABELS)
print("Random seed:", SEED)
print("Score type:", SCORE_TYPE)
print("Default threshold:", DEFAULT_THRESHOLD)
print("Results directory:", RESULTS_DIR)
print("Model directory:", MODELS_DIR)

print("Previous experiments verified:",
      len(previous_run_verification_df))

print("Matching environment packages:",
      environment_comparison_df['Version Match'].sum(),
      "/", len(environment_df))

print("Shared dataset fingerprints: Consistent")
print("Shared evaluation module: Verified")
print("Signed margin evaluation: Verified")
print("Feature artifact manifest: Verified")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nLinear SVM environment initialized successfully.")

In [ ]:
# Load and verify the fixed experimental datasets

dataset_manifest = pd.read_csv(
    SPLITS_DIR / 'prepared_dataset_manifest.csv'
)

feature_manifest = pd.read_csv(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
)

datasets = {}
dataset_records = {}
verification_records = []

for partition, source_partition, expected_hash in [
    ('train', 'train_representative', EXPECTED_TRAINING_HASH),
    ('validation', 'validation', EXPECTED_VALIDATION_HASH)
]:

    record = dataset_manifest.loc[
        dataset_manifest['Partition'] == source_partition
    ].iloc[0]

    dataset_path = PROJECT_ROOT / record['Dataset Path']
    dataset_hash = calculate_sha256(dataset_path)

    assert dataset_hash == record['SHA256']
    assert dataset_hash == expected_hash

    df = pd.read_parquet(dataset_path)

    assert len(df) == record['Rows']
    assert df['id'].is_unique
    assert df[LABELS].isin([0, 1]).all().all()
    assert df['comment_text'].notna().all()

    # Verify feature row alignment

    row_record = feature_manifest.loc[
        (feature_manifest['Artifact Type'] == 'Row Manifest')
        & (feature_manifest['Partition'] == partition)
    ].iloc[0]

    row_path = PROJECT_ROOT / row_record['Path']

    assert calculate_sha256(row_path) == row_record['SHA256']
    assert row_record['Source Dataset SHA256'] == dataset_hash

    row_manifest = pd.read_csv(
        row_path,
        dtype={'id': str}
    )

    assert row_manifest['id'].tolist() == df['id'].tolist()

    assert (
        row_manifest['group_id'].tolist()
        == df['group_id'].tolist()
    )

    assert np.array_equal(
        row_manifest['row_index'].to_numpy(),
        np.arange(len(df))
    )

    datasets[partition] = df
    dataset_records[partition] = record

    verification_records.append({
        'Partition': partition,
        'Comments': len(df),
        'Unique IDs': df['id'].nunique(),
        'Dataset Hash Verified': True,
        'Previous Experiments Match': True,
        'Row Manifest Verified': True,
        'Row Alignment Verified': True,
        'Valid Labels': True
    })

train_df = datasets['train']
val_df = datasets['validation']

y_train = train_df[LABELS].to_numpy(dtype=np.int8)
y_val = val_df[LABELS].to_numpy(dtype=np.int8)

assert y_train.shape == (59997, len(LABELS))
assert y_val.shape == (31911, len(LABELS))

assert set(train_df['id']).isdisjoint(val_df['id'])

assert set(train_df['group_id']).isdisjoint(
    val_df['group_id']
)

assert (y_train.sum(axis=0) > 0).all()
assert (y_val.sum(axis=0) > 0).all()

dataset_verification_df = pd.DataFrame(
    verification_records
)

print("Dataset integrity verification")

display(dataset_verification_df)

dataset_verification_df.to_csv(
    TABLES_DIR / 'dataset_verification.csv',
    index=False
)


# Define a reusable sparse feature loader

def load_verified_features(representation):

    assert representation in REPRESENTATIONS

    matrices = {}
    records = []

    for partition in ['train', 'validation']:

        record = feature_manifest.loc[
            (feature_manifest['Artifact Type'] == 'Sparse Matrix')
            & (feature_manifest['Representation'] == representation)
            & (feature_manifest['Partition'] == partition)
        ].iloc[0]

        path = PROJECT_ROOT / record['Path']

        assert calculate_sha256(path) == record['SHA256']

        assert (
            record['Source Dataset SHA256']
            == dataset_records[partition]['SHA256']
        )

        matrix = sp.load_npz(path)

        assert matrix.format == 'csr'
        assert matrix.dtype == np.float32

        assert matrix.shape == (
            len(datasets[partition]),
            FEATURE_BUDGET
        )

        assert np.isfinite(matrix.data).all()
        assert (matrix.data >= 0).all()

        memory_bytes = (
            matrix.data.nbytes
            + matrix.indices.nbytes
            + matrix.indptr.nbytes
        )

        matrices[partition] = matrix

        records.append({
            'Representation': representation,
            'Partition': partition,
            'Comments': matrix.shape[0],
            'Features': matrix.shape[1],
            'Nonzero Elements': matrix.nnz,
            'Data Type': str(matrix.dtype),
            'Sparse Memory (MiB)': memory_bytes / (1024 ** 2),
            'Zero Feature Comments':
                int(np.sum(np.diff(matrix.indptr) == 0)),
            'Feature Hash Verified': True,
            'Source Dataset Verified': True,
            'Verification': 'Passed'
        })

    return (
        matrices['train'],
        matrices['validation'],
        pd.DataFrame(records)
    )


# Verify each representation without retaining all matrices

feature_verification_records = []

for representation in REPRESENTATIONS:

    print(f"\nVerifying {representation}...")

    X_train_temp, X_val_temp, verification = (
        load_verified_features(representation)
    )

    feature_verification_records.append(verification)

    print(
        "Training:", X_train_temp.shape,
        "| Validation:", X_val_temp.shape
    )

    del X_train_temp, X_val_temp
    gc.collect()

feature_verification_df = pd.concat(
    feature_verification_records,
    ignore_index=True
)

print("\nFeature representation verification")

display(feature_verification_df.round(4))

feature_verification_df.to_csv(
    TABLES_DIR / 'feature_representation_verification.csv',
    index=False
)


# Compare feature properties with Logistic Regression

lr_feature_verification_df = pd.read_csv(
    CLASSICAL_DIR / '02_logistic_regression'
    / 'results' / 'tables'
    / 'feature_representation_verification.csv'
)

comparison_columns = [
    'Representation',
    'Partition',
    'Comments',
    'Features',
    'Nonzero Elements',
    'Data Type',
    'Sparse Memory (MiB)',
    'Zero Feature Comments'
]

pd.testing.assert_frame_equal(
    feature_verification_df[comparison_columns]
    .reset_index(drop=True),

    lr_feature_verification_df[comparison_columns]
    .reset_index(drop=True),

    check_exact=False,
    rtol=1e-12,
    atol=1e-12
)


# Record label support and class imbalance

label_support_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': y_train.sum(axis=0),
    'Training Negatives': len(y_train) - y_train.sum(axis=0),
    'Validation Positives': y_val.sum(axis=0),
    'Validation Negatives': len(y_val) - y_val.sum(axis=0)
})

label_support_df['Training Imbalance Ratio'] = (
    label_support_df['Training Negatives']
    / label_support_df['Training Positives']
)

print("\nLabel support and class imbalance")

display(label_support_df.round(4))

label_support_df.to_csv(
    TABLES_DIR / 'label_support_and_imbalance.csv',
    index=False
)


# Summarize dataset and feature verification

print("\nDataset and feature verification summary")

print("Training label shape:", y_train.shape)
print("Validation label shape:", y_val.shape)
print("Feature representations verified:", len(REPRESENTATIONS))
print("Sparse matrices verified:", len(feature_verification_df))
print("Consistency with previous experiments: Verified")
print("Consistency with LR feature properties: Verified")
print("Training-validation ID overlap: 0")
print("Training-validation group overlap: 0")
print("Reusable feature loader: Ready")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nDataset and feature verification completed successfully.")

In [ ]:
# Define the fixed Linear SVM experimental configuration

SVM_PARAMS = {
    'penalty': 'l2',
    'loss': 'squared_hinge',
    'dual': False,
    'C': 1.0,
    'tol': 1e-4,
    'max_iter': 1000,
    'fit_intercept': True,
    'intercept_scaling': 1.0,
    'class_weight': None,
    'random_state': SEED
}

OVR_PARAMS = {
    'n_jobs': 1
}

assert len(y_train) > FEATURE_BUDGET
assert DEFAULT_THRESHOLD == 0.0

# Define the three unweighted representation experiments

experiment_specs = [
    {
        'run_id': 'SVM_WORD_BASELINE_S42',
        'representation': 'word_tfidf',
        'class_weight': None,
        'purpose': (
            'Compare Linear SVM against other classical models '
            'using the common word TF-IDF representation'
        )
    },
    {
        'run_id': 'SVM_CHAR_BASELINE_S42',
        'representation': 'character_tfidf',
        'class_weight': None,
        'purpose': (
            'Investigate character-level representation effectiveness'
        )
    },
    {
        'run_id': 'SVM_COMBINED_BASELINE_S42',
        'representation': 'combined_tfidf',
        'class_weight': None,
        'purpose': (
            'Investigate word and character feature fusion '
            'under a fixed feature budget'
        )
    }
]

experiment_plan_df = pd.DataFrame([
    {
        'Run ID': spec['run_id'],
        'Representation': spec['representation'],
        'Class Weight': 'None',
        'Training Samples': len(y_train),
        'Features': FEATURE_BUDGET,
        'Purpose': spec['purpose']
    }
    for spec in experiment_specs
])

experiment_plan_df = pd.concat([
    experiment_plan_df,
    pd.DataFrame([{
        'Run ID': 'To be selected',
        'Representation': 'Best unweighted SVM representation',
        'Class Weight': 'balanced',
        'Training Samples': len(y_train),
        'Features': FEATURE_BUDGET,
        'Purpose': (
            'Controlled investigation of class imbalance handling'
        )
    }])
], ignore_index=True)

print("Linear SVM experiment plan")

display(experiment_plan_df)

experiment_plan_df.to_csv(
    TABLES_DIR / 'svm_experiment_plan.csv',
    index=False
)


# Document the rationale for each hyperparameter

hyperparameter_rationale_df = pd.DataFrame([
    {
        'Parameter': 'Estimator',
        'Value': 'LinearSVC',
        'Justification': (
            'Efficient linear classification for high-dimensional '
            'sparse text features'
        )
    },
    {
        'Parameter': 'Classification Strategy',
        'Value': 'One-vs-Rest',
        'Justification': (
            'Independent binary classification for each toxicity label'
        )
    },
    {
        'Parameter': 'penalty',
        'Value': 'l2',
        'Justification': (
            'Regularizes model coefficients and provides a '
            'consistent baseline'
        )
    },
    {
        'Parameter': 'loss',
        'Value': 'squared_hinge',
        'Justification': (
            'Standard margin-based LinearSVC loss that penalizes '
            'larger margin violations quadratically'
        )
    },
    {
        'Parameter': 'dual',
        'Value': 'False',
        'Justification': (
            'Primal optimization is appropriate because the '
            'number of training samples exceeds feature dimensions'
        )
    },
    {
        'Parameter': 'C',
        'Value': '1.0',
        'Justification': (
            'Fixed initial regularization strength; not selected '
            'through hyperparameter search'
        )
    },
    {
        'Parameter': 'tol',
        'Value': '0.0001',
        'Justification': (
            'Standard convergence tolerance'
        )
    },
    {
        'Parameter': 'max_iter',
        'Value': '1000',
        'Justification': (
            'Initial iteration budget with explicit convergence '
            'verification for every binary classifier'
        )
    },
    {
        'Parameter': 'fit_intercept',
        'Value': 'True',
        'Justification': (
            'Allows the decision boundary to include an intercept'
        )
    },
    {
        'Parameter': 'intercept_scaling',
        'Value': '1.0',
        'Justification': (
            'Retains the standard LIBLINEAR intercept scaling; '
            'the synthetic intercept feature is regularized'
        )
    },
    {
        'Parameter': 'class_weight',
        'Value': 'None / balanced',
        'Justification': (
            'Isolates the effect of inverse-frequency class weighting '
            'without changing training sample composition'
        )
    },
    {
        'Parameter': 'random_state',
        'Value': str(SEED),
        'Justification': (
            'Retained for configuration consistency; primal '
            'LinearSVC optimization does not use this seed'
        )
    },
    {
        'Parameter': 'n_jobs',
        'Value': '1',
        'Justification': (
            'Sequential binary classifier training to limit '
            'concurrent memory consumption'
        )
    },
    {
        'Parameter': 'Default Threshold',
        'Value': '0.0',
        'Justification': (
            'Signed decision margin relative to the separating '
            'hyperplane'
        )
    },
    {
        'Parameter': 'Selection Metric',
        'Value': 'Macro Average Precision',
        'Justification': (
            'Evaluates ranking performance independently for each '
            'label and gives equal importance to all six labels'
        )
    }
])

print("\nHyperparameter rationale")

display(hyperparameter_rationale_df)

hyperparameter_rationale_df.to_csv(
    TABLES_DIR / 'svm_hyperparameter_rationale.csv',
    index=False
)


# Calculate balanced class weights using training labels only

positive_counts = y_train.sum(axis=0)
negative_counts = len(y_train) - positive_counts

assert (positive_counts > 0).all()
assert (negative_counts > 0).all()

balanced_positive_weights = (
    len(y_train) / (2 * positive_counts)
)

balanced_negative_weights = (
    len(y_train) / (2 * negative_counts)
)

balanced_weights_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': positive_counts,
    'Training Negatives': negative_counts,
    'Negative Class Weight': balanced_negative_weights,
    'Positive Class Weight': balanced_positive_weights,
    'Positive-to-Negative Weight Ratio': (
        balanced_positive_weights
        / balanced_negative_weights
    )
})

# Verify equal total weighted contributions for both classes

assert np.allclose(
    positive_counts * balanced_positive_weights,
    len(y_train) / 2
)

assert np.allclose(
    negative_counts * balanced_negative_weights,
    len(y_train) / 2
)

print("\nExpected balanced class weights")

display(balanced_weights_df.round(6))

balanced_weights_df.to_csv(
    TABLES_DIR / 'svm_expected_balanced_class_weights.csv',
    index=False
)


# Define a reusable model constructor

def build_svm_model(class_weight=None):

    assert class_weight in [None, 'balanced']

    params = SVM_PARAMS.copy()
    params['class_weight'] = class_weight

    estimator = LinearSVC(**params)

    model = OneVsRestClassifier(
        estimator,
        **OVR_PARAMS
    )

    return model


# Verify model construction without fitting

unweighted_svm = build_svm_model()

balanced_svm = build_svm_model(
    class_weight='balanced'
)

unweighted_params = unweighted_svm.estimator.get_params()
balanced_params = balanced_svm.estimator.get_params()

different_parameters = [
    parameter
    for parameter in unweighted_params
    if unweighted_params[parameter]
    != balanced_params[parameter]
]

assert different_parameters == ['class_weight']

assert unweighted_svm.n_jobs == balanced_svm.n_jobs == 1
assert unweighted_params['dual'] is False

assert hasattr(
    unweighted_svm.estimator,
    'decision_function'
)

assert not hasattr(
    unweighted_svm.estimator,
    'predict_proba'
)

print("\nControlled model configuration verified")
print("Changed parameters:", different_parameters)


# Save the experimental protocol before training

svm_experiment_config = {
    'notebook': '03_linear_svm',
    'model': MODEL_NAME,
    'classification_strategy': 'One-vs-Rest',
    'labels': LABELS,
    'num_labels': len(LABELS),
    'seed': SEED,
    'training_samples': len(y_train),
    'validation_samples': len(y_val),
    'sampling_strategy': 'representative_multilabel',
    'feature_budget': FEATURE_BUDGET,
    'model_parameters': SVM_PARAMS,
    'wrapper_parameters': OVR_PARAMS,
    'experiments': experiment_specs,
    'score_type': SCORE_TYPE,
    'default_threshold': DEFAULT_THRESHOLD,
    'threshold_comparison_operator': '>=',
    'primary_selection_metric': 'macro_average_precision',
    'secondary_selection_metric': 'optimized_macro_f1',
    'threshold_optimization_metric': 'per_label_f1',
    'threshold_tie_breaking': 'highest_threshold',
    'probability_calibration': False,
    'micro_ranking_metric_limitation': (
        'Margins from independent binary classifiers may have '
        'different scales, limiting interpretation of pooled '
        'micro ranking metrics'
    ),
    'weighted_experiment': {
        'class_weight': 'balanced',
        'representation_selection': 'validation_macro_ap',
        'other_parameters': 'unchanged',
        'scope': 'controlled_extension',
        'status': 'planned'
    },
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
    'feature_manifest_sha256': feature_manifest_hash,
    'evaluation_module_sha256': evaluation_hash,
    'sklearn_version': sklearn.__version__,
    'official_test_used': False
}

SVM_EXPERIMENT_CONFIG_PATH = (
    RUN_CONFIGS_DIR / 'svm_experiment_config.json'
)

with open(SVM_EXPERIMENT_CONFIG_PATH, 'w') as file:
    json.dump(
        svm_experiment_config,
        file,
        indent=4
    )

with open(SVM_EXPERIMENT_CONFIG_PATH, 'r') as file:
    saved_config = json.load(file)

assert saved_config == svm_experiment_config

del unweighted_svm, balanced_svm

# Initialize storage for completed experiments

svm_results = {}

print("\nLinear SVM configuration summary")

print("Estimator: LinearSVC")
print("Classification strategy: One-vs-Rest")
print("Loss:", SVM_PARAMS['loss'])
print("Penalty:", SVM_PARAMS['penalty'])
print("C:", SVM_PARAMS['C'])
print("Optimization: Primal")
print("Maximum iterations:", SVM_PARAMS['max_iter'])
print("Training samples:", len(y_train))
print("Feature dimensions:", FEATURE_BUDGET)
print("Planned unweighted experiments:", len(experiment_specs))
print("Planned balanced experiments: 1")
print("Primary selection metric: Macro AP")
print("Threshold optimization: Per-label F1")
print("Default threshold:", DEFAULT_THRESHOLD)
print("Score type:", SCORE_TYPE)
print("Controlled weighting configuration: Verified")
print("Experiment configuration: Saved and verified")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nLinear SVM experiment configuration completed successfully.")

In [ ]:
# Define a reusable training and evaluation pipeline for Linear SVM

def run_svm_experiment(spec):

    run_id = spec['run_id']
    representation = spec['representation']
    class_weight = spec['class_weight']

    assert representation in REPRESENTATIONS
    assert class_weight in [None, 'balanced']

    run_dir = RESULTS_DIR / run_id
    run_tables_dir = run_dir / 'tables'
    run_predictions_dir = run_dir / 'predictions'
    run_logs_dir = run_dir / 'logs'

    for folder in [
        run_tables_dir,
        run_predictions_dir,
        run_logs_dir
    ]:
        folder.mkdir(parents=True, exist_ok=True)

    model_path = MODELS_DIR / f'{run_id}.joblib'
    config_path = RUN_CONFIGS_DIR / f'{run_id}.json'

    threshold_path = (
        RUN_CONFIGS_DIR / f'{run_id}_thresholds.json'
    )

    predictions_path = (
        run_predictions_dir / 'validation_predictions.parquet'
    )

    print(f"\nStarting experiment: {run_id}")
    print("Representation:", representation)
    print("Class weight:", class_weight)

    start_time = datetime.now(timezone.utc).isoformat()

    # Load one verified sparse representation at a time

    loading_start = time.perf_counter()

    X_train, X_val, feature_checks = (
        load_verified_features(representation)
    )

    feature_loading_time = (
        time.perf_counter() - loading_start
    )

    feature_checks.to_csv(
        run_tables_dir / 'feature_verification.csv',
        index=False
    )

    assert X_train.shape == (
        len(y_train), FEATURE_BUDGET
    )

    assert X_val.shape == (
        len(y_val), FEATURE_BUDGET
    )

    print("Training features:", X_train.shape)
    print("Validation features:", X_val.shape)

    # Train six independent binary classifiers

    model = build_svm_model(class_weight)

    training_start = time.perf_counter()

    with warnings.catch_warnings(record=True) as captured_warnings:

        warnings.simplefilter('always')

        model.fit(X_train, y_train)

    training_time = time.perf_counter() - training_start

    warning_records = [
        {
            'Category': warning.category.__name__,
            'Message': str(warning.message)
        }
        for warning in captured_warnings
    ]

    convergence_warnings = sum(
        issubclass(warning.category, ConvergenceWarning)
        for warning in captured_warnings
    )

    with open(
        run_logs_dir / 'training_warnings.json', 'w'
    ) as file:
        json.dump(warning_records, file, indent=4)

    assert convergence_warnings == 0, (
        f"Convergence warnings detected in {run_id}"
    )

    assert len(model.estimators_) == len(LABELS)
    assert model.multilabel_

    assert np.array_equal(
        model.classes_,
        np.arange(len(LABELS))
    )

    # Verify each binary classifier

    classifier_records = []

    for label, estimator in zip(
        LABELS, model.estimators_
    ):

        assert np.array_equal(
            estimator.classes_, [0, 1]
        )

        assert estimator.coef_.shape == (
            1, FEATURE_BUDGET
        )

        assert np.isfinite(estimator.coef_).all()
        assert np.isfinite(estimator.intercept_).all()

        iterations = int(estimator.n_iter_)

        assert iterations < SVM_PARAMS['max_iter'], (
            f"Iteration limit reached for {label}"
        )

        classifier_records.append({
            'Label': label,
            'Iterations': iterations,
            'Maximum Iterations': SVM_PARAMS['max_iter'],
            'Coefficient L2 Norm': float(
                np.linalg.norm(estimator.coef_)
            ),
            'Intercept': float(estimator.intercept_[0]),
            'Convergence': 'Passed'
        })

    classifier_summary_df = pd.DataFrame(
        classifier_records
    )

    classifier_summary_df.to_csv(
        run_tables_dir / 'classifier_training_summary.csv',
        index=False
    )

    print("\nClassifier training summary")

    display(classifier_summary_df.round(6))


    # Generate signed validation decision scores

    prediction_start = time.perf_counter()

    validation_scores = np.asarray(
        model.decision_function(X_val),
        dtype=np.float64
    )

    prediction_time = (
        time.perf_counter() - prediction_start
    )

    assert validation_scores.shape == y_val.shape
    assert np.isfinite(validation_scores).all()

    # Confirm the score columns match binary classifiers

    alignment_sample = min(100, len(y_val))

    for i, estimator in enumerate(model.estimators_):

        individual_scores = estimator.decision_function(
            X_val[:alignment_sample]
        )

        assert np.allclose(
            validation_scores[:alignment_sample, i],
            individual_scores,
            rtol=0,
            atol=1e-10
        )

    # Record score distributions without probability conversion

    score_distribution_df = pd.DataFrame([
        {
            'Label': label,
            'Minimum Score': validation_scores[:, i].min(),
            'Maximum Score': validation_scores[:, i].max(),
            'Mean Score': validation_scores[:, i].mean(),
            'Standard Deviation': validation_scores[:, i].std(),
            'Median Score': np.median(validation_scores[:, i]),
            'Negative Scores': int(
                (validation_scores[:, i] < 0).sum()
            ),
            'Exact Zero Scores': int(
                (validation_scores[:, i] == 0).sum()
            ),
            'Positive Scores': int(
                (validation_scores[:, i] > 0).sum()
            )
        }
        for i, label in enumerate(LABELS)
    ])

    score_distribution_df.to_csv(
        run_tables_dir / 'decision_score_distribution.csv',
        index=False
    )


    # Evaluate the standard zero decision threshold

    default_predictions = apply_thresholds(
        validation_scores,
        DEFAULT_THRESHOLD
    )

    default_metrics, default_per_label_df = (
        evaluate_multilabel(
            y_true=y_val,
            y_scores=validation_scores,
            labels=LABELS,
            thresholds=DEFAULT_THRESHOLD
        )
    )

    # Optimize each classification threshold on validation data

    threshold_df = optimize_thresholds(
        y_true=y_val,
        y_scores=validation_scores,
        labels=LABELS
    )

    assert threshold_df['Label'].tolist() == LABELS

    optimized_thresholds = threshold_df[
        'Optimal Threshold'
    ].to_numpy(dtype=np.float64)

    assert len(optimized_thresholds) == len(LABELS)
    assert np.isfinite(optimized_thresholds).all()

    optimized_predictions = apply_thresholds(
        validation_scores,
        optimized_thresholds
    )

    optimized_metrics, optimized_per_label_df = (
        evaluate_multilabel(
            y_true=y_val,
            y_scores=validation_scores,
            labels=LABELS,
            thresholds=optimized_thresholds
        )
    )

    # Verify threshold optimization and ranking invariance

    assert np.all(
        optimized_per_label_df['F1'].to_numpy()
        >= default_per_label_df['F1'].to_numpy() - 1e-12
    )

    for metric in [
        'Macro AP',
        'Micro AP',
        'Macro ROC-AUC',
        'Micro ROC-AUC'
    ]:

        assert np.isclose(
            default_metrics[metric],
            optimized_metrics[metric],
            rtol=0,
            atol=1e-12
        )


    # Save overall and per-label evaluation results

    pd.DataFrame([default_metrics]).to_csv(
        run_tables_dir / 'default_overall_metrics.csv',
        index=False
    )

    pd.DataFrame([optimized_metrics]).to_csv(
        run_tables_dir / 'optimized_overall_metrics.csv',
        index=False
    )

    default_per_label_df.to_csv(
        run_tables_dir / 'default_per_label_metrics.csv',
        index=False
    )

    optimized_per_label_df.to_csv(
        run_tables_dir / 'optimized_per_label_metrics.csv',
        index=False
    )

    threshold_df.to_csv(
        run_tables_dir / 'optimized_thresholds.csv',
        index=False
    )

    overall_comparison_df = pd.DataFrame([
        {
            'Metric': metric,
            'Default': default_metrics[metric],
            'Optimized': optimized_metrics[metric],
            'Change': (
                optimized_metrics[metric]
                - default_metrics[metric]
            )
        }
        for metric in default_metrics
    ])

    overall_comparison_df.to_csv(
        run_tables_dir / 'overall_threshold_comparison.csv',
        index=False
    )

    print("\nOverall validation performance")

    display(overall_comparison_df.round(6))


    # Compare per-label threshold effects

    per_label_comparison_df = pd.DataFrame({
        'Label': LABELS,
        'Average Precision':
            optimized_per_label_df['Average Precision'],
        'ROC-AUC':
            optimized_per_label_df['ROC-AUC'],
        'Default Threshold': DEFAULT_THRESHOLD,
        'Optimized Threshold': optimized_thresholds,
        'Default Precision':
            default_per_label_df['Precision'],
        'Optimized Precision':
            optimized_per_label_df['Precision'],
        'Default Recall':
            default_per_label_df['Recall'],
        'Optimized Recall':
            optimized_per_label_df['Recall'],
        'Default F1':
            default_per_label_df['F1'],
        'Optimized F1':
            optimized_per_label_df['F1'],
        'Actual Positives':
            optimized_per_label_df['Positive Support']
    })

    per_label_comparison_df['F1 Improvement'] = (
        per_label_comparison_df['Optimized F1']
        - per_label_comparison_df['Default F1']
    )

    per_label_comparison_df.to_csv(
        run_tables_dir / 'per_label_threshold_comparison.csv',
        index=False
    )

    print("\nPer-label performance comparison")

    display(per_label_comparison_df.round(6))


    # Save full-precision decision thresholds

    threshold_config = {
        'run_id': run_id,
        'labels': LABELS,
        'score_type': SCORE_TYPE,
        'default_threshold': DEFAULT_THRESHOLD,
        'optimized_thresholds': {
            label: float(threshold)
            for label, threshold in zip(
                LABELS, optimized_thresholds
            )
        },
        'selection_partition': 'validation',
        'optimization_metric': 'per_label_f1',
        'tie_breaking': 'highest_threshold',
        'comparison_operator': '>=',
        'probability_calibration': False,
        'official_test_used': False
    }

    with open(threshold_path, 'w') as file:
        json.dump(threshold_config, file, indent=4)


    # Save validation scores and binary predictions

    predictions_df = pd.DataFrame({
        'id': val_df['id'].to_numpy()
    })

    for i, label in enumerate(LABELS):

        predictions_df[f'{label}_true'] = y_val[:, i]

        predictions_df[f'{label}_score'] = (
            validation_scores[:, i]
        )

        predictions_df[f'{label}_pred_default'] = (
            default_predictions[:, i].astype(np.int8)
        )

        predictions_df[f'{label}_pred_optimized'] = (
            optimized_predictions[:, i].astype(np.int8)
        )

    assert predictions_df.shape == (
        len(y_val), 1 + 4 * len(LABELS)
    )

    predictions_df.to_parquet(
        predictions_path,
        index=False
    )


    # Save the trained model and experiment configuration

    joblib.dump(
        model,
        model_path,
        compress=3
    )

    actual_model_params = SVM_PARAMS.copy()
    actual_model_params['class_weight'] = class_weight

    run_config = {
        'run_id': run_id,
        'notebook': '03_linear_svm',
        'model': MODEL_NAME,
        'feature_representation': representation,
        'feature_count': FEATURE_BUDGET,
        'classification_strategy': 'One-vs-Rest',
        'labels': LABELS,
        'seed': SEED,
        'sampling_strategy': 'representative_multilabel',
        'training_samples': len(y_train),
        'validation_samples': len(y_val),
        'model_parameters': actual_model_params,
        'wrapper_parameters': OVR_PARAMS,
        'score_type': SCORE_TYPE,
        'default_threshold': DEFAULT_THRESHOLD,
        'primary_selection_metric':
            'macro_average_precision',
        'threshold_optimization_metric': 'per_label_f1',
        'training_dataset_sha256': EXPECTED_TRAINING_HASH,
        'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
        'feature_manifest_sha256': feature_manifest_hash,
        'evaluation_module_sha256': evaluation_hash,
        'threshold_path': str(
            threshold_path.relative_to(PROJECT_ROOT)
        ),
        'model_path': str(
            model_path.relative_to(PROJECT_ROOT)
        ),
        'sklearn_version': sklearn.__version__,
        'probability_calibration': False,
        'official_test_used': False
    }

    with open(config_path, 'w') as file:
        json.dump(run_config, file, indent=4)

    model_size_mib = (
        model_path.stat().st_size / (1024 ** 2)
    )


    # Verify saved predictions and threshold precision

    saved_predictions_df = pd.read_parquet(
        predictions_path
    )

    pd.testing.assert_frame_equal(
        predictions_df,
        saved_predictions_df,
        check_exact=True
    )

    with open(threshold_path, 'r') as file:
        saved_threshold_config = json.load(file)

    saved_thresholds = np.array([
        saved_threshold_config['optimized_thresholds'][label]
        for label in LABELS
    ], dtype=np.float64)

    assert np.array_equal(
        optimized_thresholds,
        saved_thresholds
    )

    with open(config_path, 'r') as file:
        saved_run_config = json.load(file)

    assert saved_run_config == run_config


    # Reload the model and reproduce all validation scores

    del model, X_train
    gc.collect()

    reloaded_model = joblib.load(model_path)

    reconstructed_scores = np.asarray(
        reloaded_model.decision_function(X_val),
        dtype=np.float64
    )

    assert reconstructed_scores.shape == validation_scores.shape

    reconstruction_error = float(
        np.max(
            np.abs(
                reconstructed_scores - validation_scores
            )
        )
    )

    assert reconstruction_error <= 1e-12

    assert np.array_equal(
        apply_thresholds(
            reconstructed_scores,
            DEFAULT_THRESHOLD
        ),
        default_predictions
    )

    assert np.array_equal(
        apply_thresholds(
            reconstructed_scores,
            saved_thresholds
        ),
        optimized_predictions
    )


    # Reconstruct metrics from saved validation predictions

    saved_targets = saved_predictions_df[[
        f'{label}_true' for label in LABELS
    ]].to_numpy()

    saved_scores = saved_predictions_df[[
        f'{label}_score' for label in LABELS
    ]].to_numpy()

    assert np.array_equal(saved_targets, y_val)

    reconstructed_default, reconstructed_default_labels = (
        evaluate_multilabel(
            saved_targets,
            saved_scores,
            LABELS,
            thresholds=DEFAULT_THRESHOLD
        )
    )

    reconstructed_optimized, reconstructed_optimized_labels = (
        evaluate_multilabel(
            saved_targets,
            saved_scores,
            LABELS,
            thresholds=saved_thresholds
        )
    )

    for original, reconstructed in [
        (default_metrics, reconstructed_default),
        (optimized_metrics, reconstructed_optimized)
    ]:

        for metric in original:

            assert np.isclose(
                original[metric],
                reconstructed[metric],
                rtol=0,
                atol=1e-12
            )

    pd.testing.assert_frame_equal(
        default_per_label_df,
        reconstructed_default_labels,
        check_exact=False,
        rtol=0,
        atol=1e-12
    )

    pd.testing.assert_frame_equal(
        optimized_per_label_df,
        reconstructed_optimized_labels,
        check_exact=False,
        rtol=0,
        atol=1e-12
    )


    # Save the experiment summary and reproducibility log

    end_time = datetime.now(timezone.utc).isoformat()

    run_summary = {
        'Run ID': run_id,
        'Model': MODEL_NAME,
        'Feature Representation': representation,
        'Class Weight': (
            'None' if class_weight is None else class_weight
        ),
        'Training Samples': len(y_train),
        'Validation Samples': len(y_val),
        'Features': FEATURE_BUDGET,
        'Macro AP': default_metrics['Macro AP'],
        'Micro AP': default_metrics['Micro AP'],
        'Macro ROC-AUC': default_metrics['Macro ROC-AUC'],
        'Default Macro F1': default_metrics['Macro F1'],
        'Optimized Macro F1': optimized_metrics['Macro F1'],
        'Default Micro F1': default_metrics['Micro F1'],
        'Optimized Micro F1': optimized_metrics['Micro F1'],
        'Optimized Macro Precision':
            optimized_metrics['Macro Precision'],
        'Optimized Macro Recall':
            optimized_metrics['Macro Recall'],
        'Training Time (s)': training_time,
        'Validation Prediction Time (s)': prediction_time,
        'Feature Loading Time (s)': feature_loading_time,
        'Model Size (MiB)': model_size_mib,
        'Maximum Score Reconstruction Error':
            reconstruction_error,
        'Exact Zero Decision Scores': int(
            (validation_scores == 0).sum()
        ),
        'Convergence Warnings': convergence_warnings,
        'Official Test Used': False,
        'Status': 'Completed'
    }

    pd.DataFrame([run_summary]).to_csv(
        run_tables_dir / 'run_summary.csv',
        index=False
    )

    training_log = {
        'run_id': run_id,
        'start_time': start_time,
        'end_time': end_time,
        'training_time_seconds': training_time,
        'validation_prediction_time_seconds': prediction_time,
        'feature_loading_time_seconds': feature_loading_time,
        'convergence_warnings': convergence_warnings,
        'all_classifiers_converged': True,
        'score_alignment_verified': True,
        'threshold_roundtrip_verified': True,
        'prediction_roundtrip_verified': True,
        'model_reconstruction_verified': True,
        'metric_reconstruction_verified': True,
        'maximum_score_reconstruction_error':
            reconstruction_error,
        'exact_zero_decision_scores': int(
            (validation_scores == 0).sum()
        ),
        'official_test_used': False,
        'status': 'Completed'
    }

    training_log_path = run_logs_dir / 'training_log.json'

    with open(training_log_path, 'w') as file:
        json.dump(training_log, file, indent=4)


    # Fingerprint critical experiment artifacts

    critical_artifacts = {
        'Model': model_path,
        'Run Configuration': config_path,
        'Threshold Configuration': threshold_path,
        'Validation Predictions': predictions_path,
        'Training Log': training_log_path,
        'Feature Manifest':
            FEATURES_DIR / 'feature_artifact_manifest.csv',
        'Evaluation Module':
            SRC_DIR / 'evaluation_utils.py'
    }

    artifact_records = []

    for artifact_type, path in critical_artifacts.items():

        assert path.is_file()
        assert path.stat().st_size > 0

        artifact_records.append({
            'Run ID': run_id,
            'Artifact Type': artifact_type,
            'Path': str(path.relative_to(PROJECT_ROOT)),
            'Size Bytes': path.stat().st_size,
            'SHA256': calculate_sha256(path)
        })

    artifact_manifest_df = pd.DataFrame(
        artifact_records
    )

    artifact_manifest_df.to_csv(
        run_tables_dir / 'artifact_manifest.csv',
        index=False
    )


    # Prepare the registry record without updating the registry yet

    registry_record = {
        'run_id': run_id,
        'notebook': '03_linear_svm',
        'model': MODEL_NAME,
        'feature_representation': representation,
        'loss_function': 'Squared hinge loss',
        'sampling_strategy': 'representative_multilabel',
        'training_samples': len(y_train),
        'seed': SEED,
        'max_length': '',
        'config_path': str(
            config_path.relative_to(PROJECT_ROOT)
        ),
        'model_path': str(
            model_path.relative_to(PROJECT_ROOT)
        ),
        'results_path': str(
            run_dir.relative_to(PROJECT_ROOT)
        ),
        'validation_macro_ap': default_metrics['Macro AP'],
        'validation_macro_f1': optimized_metrics['Macro F1'],
        'training_time_seconds': training_time,
        'status': 'Completed',
        'start_time': start_time,
        'end_time': end_time,
        'notes': (
            f"Class weight={class_weight}; C=1.0; "
            "L2 squared hinge; primal optimization; "
            "signed margins; per-label validation thresholds; "
            "pooled micro ranking metrics interpreted cautiously; "
            "official test not used."
        )
    }

    assert set(registry_record.keys()) == set(
        registry_df.columns
    )


    # Retain compact results for subsequent comparisons

    result = {
        'spec': spec.copy(),
        'summary': run_summary,
        'default_metrics': default_metrics,
        'optimized_metrics': optimized_metrics,
        'default_per_label': default_per_label_df,
        'optimized_per_label': optimized_per_label_df,
        'thresholds': threshold_df,
        'registry_record': registry_record,
        'predictions_path': predictions_path,
        'model_path': model_path,
        'results_path': run_dir
    }

    print("\nExperiment summary")

    display(
        pd.DataFrame([run_summary]).T
    )

    print("\nModel reconstruction: Passed")
    print("Metric reconstruction: Passed")
    print("Decision score alignment: Passed")
    print("Exact zero decision scores:",
          run_summary['Exact Zero Decision Scores'])
    print("Experiment artifacts saved:", run_dir)

    # Release large matrices and model objects

    del X_val
    del reloaded_model
    del reconstructed_scores
    del validation_scores
    del predictions_df
    del saved_predictions_df
    del saved_scores
    del saved_targets
    del default_predictions
    del optimized_predictions

    gc.collect()

    print(f"\nExperiment completed: {run_id}")

    return result


# Verify that the reusable pipeline is initialized

assert callable(run_svm_experiment)
assert len(svm_results) == 0

print("Linear SVM training pipeline initialized successfully.")
print("Supported feature representations:", REPRESENTATIONS)
print("Supported class weights: None, balanced")
print("Prediction method: decision_function")
print("Default decision threshold:", DEFAULT_THRESHOLD)
print("Threshold optimization: Per-label F1")
print("Convergence verification: Enabled")
print("Artifact fingerprint verification: Enabled")
print("Model reconstruction verification: Enabled")
print("Metric reconstruction verification: Enabled")
print("Sequential feature loading: Enabled")
print("Completed SVM experiments:", len(svm_results))
print("Model training performed: No")
print("Official test dataset: Not loaded")

In [ ]:
# Train and evaluate the unweighted word-based Linear SVM

word_spec = experiment_specs[0]

assert word_spec['representation'] == 'word_tfidf'
assert word_spec['class_weight'] is None

word_result = run_svm_experiment(word_spec)

svm_results[word_spec['run_id']] = word_result


# Load completed word-based Naive Bayes and Logistic Regression results

baseline_run_ids = [
    'NB_WORD_BASELINE_S42',
    'LR_WORD_BASELINE_S42'
]

baseline_results = {}

for run_id in baseline_run_ids:

    record = registry_df.loc[
        registry_df['run_id'] == run_id
    ].iloc[0]

    run_dir = PROJECT_ROOT / record['results_path']
    run_config = previous_run_configs[run_id]

    assert run_config['feature_representation'] == 'word_tfidf'

    assert (
        run_config['training_dataset_sha256']
        == EXPECTED_TRAINING_HASH
    )

    assert (
        run_config['validation_dataset_sha256']
        == EXPECTED_VALIDATION_HASH
    )

    assert not run_config['official_test_used']

    # Support both notebook-level and individual run filenames

    def find_metric_table(filename):

        candidates = [
            run_dir / 'tables' / filename,
            run_dir / 'tables' / f'nb_{filename}'
        ]

        for path in candidates:

            if path.is_file():
                return pd.read_csv(path)

        raise FileNotFoundError(
            f"Required metric table not found: {filename}"
        )

    default_overall = find_metric_table(
        'default_overall_metrics.csv'
    )

    optimized_overall = find_metric_table(
        'optimized_overall_metrics.csv'
    )

    default_per_label = find_metric_table(
        'default_per_label_metrics.csv'
    )

    optimized_per_label = find_metric_table(
        'optimized_per_label_metrics.csv'
    )

    # Handle row-oriented and column-oriented metric tables

    def extract_overall_metrics(metrics_df):

        if {'Metric', 'Value'}.issubset(metrics_df.columns):

            return dict(zip(
                metrics_df['Metric'],
                metrics_df['Value']
            ))

        if {'Metric', 'Score'}.issubset(metrics_df.columns):

            return dict(zip(
                metrics_df['Metric'],
                metrics_df['Score']
            ))

        assert len(metrics_df) == 1

        return metrics_df.iloc[0].to_dict()

    default_metrics = extract_overall_metrics(
        default_overall
    )

    optimized_metrics = extract_overall_metrics(
        optimized_overall
    )

    assert default_per_label['Label'].tolist() == LABELS
    assert optimized_per_label['Label'].tolist() == LABELS

    assert np.array_equal(
        optimized_per_label['Positive Support'].to_numpy(),
        y_val.sum(axis=0)
    )

    assert np.isclose(
        float(default_metrics['Macro AP']),
        float(record['validation_macro_ap']),
        atol=1e-6,
        rtol=0
    )

    assert np.isclose(
        float(optimized_metrics['Macro F1']),
        float(record['validation_macro_f1']),
        atol=1e-6,
        rtol=0
    )

    baseline_results[run_id] = {
        'model': record['model'],
        'default_metrics': default_metrics,
        'optimized_metrics': optimized_metrics,
        'default_per_label': default_per_label,
        'optimized_per_label': optimized_per_label,
        'training_time': float(
            record['training_time_seconds']
        )
    }


# Add Linear SVM to the common word-feature comparison

baseline_results[word_spec['run_id']] = {
    'model': MODEL_NAME,
    'default_metrics': word_result['default_metrics'],
    'optimized_metrics': word_result['optimized_metrics'],
    'default_per_label': word_result['default_per_label'],
    'optimized_per_label': word_result['optimized_per_label'],
    'training_time': word_result['summary']['Training Time (s)']
}

comparison_records = []

for run_id, result in baseline_results.items():

    default = result['default_metrics']
    optimized = result['optimized_metrics']

    comparison_records.append({
        'Run ID': run_id,
        'Model': result['model'],
        'Representation': 'word_tfidf',
        'Macro AP': default['Macro AP'],
        'Micro AP': default['Micro AP'],
        'Macro ROC-AUC': default['Macro ROC-AUC'],
        'Default Macro F1': default['Macro F1'],
        'Optimized Macro F1': optimized['Macro F1'],
        'Default Micro F1': default['Micro F1'],
        'Optimized Micro F1': optimized['Micro F1'],
        'Optimized Macro Precision': optimized['Macro Precision'],
        'Optimized Macro Recall': optimized['Macro Recall'],
        'Optimized Hamming Loss': optimized['Hamming Loss'],
        'Training Time (s)': result['training_time']
    })

word_model_comparison_df = pd.DataFrame(
    comparison_records
)

print("\nClassical model comparison using identical word features")

display(word_model_comparison_df.round(6))

word_model_comparison_df.to_csv(
    TABLES_DIR / 'svm_word_baseline_model_comparison.csv',
    index=False
)


# Compare the three algorithms for each toxicity label

nb_per_label = baseline_results[
    'NB_WORD_BASELINE_S42'
]['optimized_per_label']

lr_per_label = baseline_results[
    'LR_WORD_BASELINE_S42'
]['optimized_per_label']

svm_per_label = word_result['optimized_per_label']

word_per_label_comparison_df = pd.DataFrame({
    'Label': LABELS,
    'Actual Positives': y_val.sum(axis=0),
    'NB AP': nb_per_label['Average Precision'],
    'LR AP': lr_per_label['Average Precision'],
    'SVM AP': svm_per_label['Average Precision'],
    'NB Optimized F1': nb_per_label['F1'],
    'LR Optimized F1': lr_per_label['F1'],
    'SVM Optimized F1': svm_per_label['F1'],
    'NB Precision': nb_per_label['Precision'],
    'LR Precision': lr_per_label['Precision'],
    'SVM Precision': svm_per_label['Precision'],
    'NB Recall': nb_per_label['Recall'],
    'LR Recall': lr_per_label['Recall'],
    'SVM Recall': svm_per_label['Recall']
})

word_per_label_comparison_df['SVM AP vs NB'] = (
    word_per_label_comparison_df['SVM AP']
    - word_per_label_comparison_df['NB AP']
)

word_per_label_comparison_df['SVM AP vs LR'] = (
    word_per_label_comparison_df['SVM AP']
    - word_per_label_comparison_df['LR AP']
)

word_per_label_comparison_df['SVM F1 vs NB'] = (
    word_per_label_comparison_df['SVM Optimized F1']
    - word_per_label_comparison_df['NB Optimized F1']
)

word_per_label_comparison_df['SVM F1 vs LR'] = (
    word_per_label_comparison_df['SVM Optimized F1']
    - word_per_label_comparison_df['LR Optimized F1']
)

print("\nPer-label classical model comparison")

display(word_per_label_comparison_df.round(6))

word_per_label_comparison_df.to_csv(
    TABLES_DIR / 'svm_word_baseline_per_label_comparison.csv',
    index=False
)


# Summarize performance differences against both baselines

svm_summary = word_result['summary']

nb_macro_ap = baseline_results[
    'NB_WORD_BASELINE_S42'
]['default_metrics']['Macro AP']

lr_macro_ap = baseline_results[
    'LR_WORD_BASELINE_S42'
]['default_metrics']['Macro AP']

lr_macro_f1 = baseline_results[
    'LR_WORD_BASELINE_S42'
]['optimized_metrics']['Macro F1']

nb_macro_f1 = baseline_results[
    'NB_WORD_BASELINE_S42'
]['optimized_metrics']['Macro F1']

print("\nWord-based Linear SVM experiment summary")

print("Macro AP:",
      round(svm_summary['Macro AP'], 6))

print("Macro AP difference vs Naive Bayes:",
      round(svm_summary['Macro AP'] - nb_macro_ap, 6))

print("Macro AP difference vs Logistic Regression:",
      round(svm_summary['Macro AP'] - lr_macro_ap, 6))

print("Default Macro F1:",
      round(svm_summary['Default Macro F1'], 6))

print("Optimized Macro F1:",
      round(svm_summary['Optimized Macro F1'], 6))

print("Optimized Macro F1 difference vs Naive Bayes:",
      round(svm_summary['Optimized Macro F1'] - nb_macro_f1, 6))

print("Optimized Macro F1 difference vs Logistic Regression:",
      round(svm_summary['Optimized Macro F1'] - lr_macro_f1, 6))

print("Labels with improved AP over Logistic Regression:",
      int(
          (word_per_label_comparison_df['SVM AP vs LR'] > 0).sum()
      ),
      "/", len(LABELS))

print("Labels with improved F1 over Logistic Regression:",
      int(
          (word_per_label_comparison_df['SVM F1 vs LR'] > 0).sum()
      ),
      "/", len(LABELS))

print("Exact zero decision scores:",
      svm_summary['Exact Zero Decision Scores'])

print("Convergence warnings:",
      svm_summary['Convergence Warnings'])

print("Completed SVM experiments:", len(svm_results))
print("Official test dataset: Not loaded")

print("\nWord-based Linear SVM experiment completed successfully.")

In [ ]:
# Train and evaluate the unweighted character-based Linear SVM

char_spec = experiment_specs[1]

assert char_spec['representation'] == 'character_tfidf'
assert char_spec['class_weight'] is None

char_result = run_svm_experiment(char_spec)

svm_results[char_spec['run_id']] = char_result


# Compare word and character representations

comparison_metrics = [
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Default Macro F1',
    'Optimized Macro F1',
    'Default Micro F1',
    'Optimized Micro F1',
    'Optimized Macro Precision',
    'Optimized Macro Recall',
    'Training Time (s)',
    'Validation Prediction Time (s)',
    'Model Size (MiB)'
]

word_char_comparison_df = pd.DataFrame([
    {
        'Metric': metric,
        'Word TF-IDF': word_result['summary'][metric],
        'Character TF-IDF': char_result['summary'][metric],
        'Difference': (
            char_result['summary'][metric]
            - word_result['summary'][metric]
        )
    }
    for metric in comparison_metrics
])

print("\nWord vs Character TF-IDF performance")

display(word_char_comparison_df.round(6))

word_char_comparison_df.to_csv(
    TABLES_DIR / 'svm_word_vs_character_comparison.csv',
    index=False
)


# Compare individual toxicity labels

word_metrics_df = word_result['optimized_per_label']
char_metrics_df = char_result['optimized_per_label']

assert word_metrics_df['Label'].tolist() == LABELS
assert char_metrics_df['Label'].tolist() == LABELS

assert np.array_equal(
    word_metrics_df['Positive Support'],
    char_metrics_df['Positive Support']
)

word_char_per_label_df = pd.DataFrame({
    'Label': LABELS,
    'Actual Positives':
        word_metrics_df['Positive Support'],
    'Word AP':
        word_metrics_df['Average Precision'],
    'Character AP':
        char_metrics_df['Average Precision'],
    'Word F1':
        word_metrics_df['F1'],
    'Character F1':
        char_metrics_df['F1'],
    'Word Precision':
        word_metrics_df['Precision'],
    'Character Precision':
        char_metrics_df['Precision'],
    'Word Recall':
        word_metrics_df['Recall'],
    'Character Recall':
        char_metrics_df['Recall']
})

for metric in ['AP', 'F1', 'Precision', 'Recall']:

    word_char_per_label_df[f'{metric} Difference'] = (
        word_char_per_label_df[f'Character {metric}']
        - word_char_per_label_df[f'Word {metric}']
    )

print("\nPer-label representation comparison")

display(word_char_per_label_df.round(6))

word_char_per_label_df.to_csv(
    TABLES_DIR / 'svm_word_vs_character_per_label.csv',
    index=False
)


# Load the character-based Logistic Regression results

lr_char_record = registry_df.loc[
    registry_df['run_id'] == 'LR_CHAR_BASELINE_S42'
].iloc[0]

lr_char_run_dir = (
    PROJECT_ROOT / lr_char_record['results_path']
)

lr_char_config = previous_run_configs[
    'LR_CHAR_BASELINE_S42'
]

assert (
    lr_char_config['feature_representation']
    == 'character_tfidf'
)

assert (
    lr_char_config['model_parameters']['class_weight']
    is None
)

assert (
    lr_char_config['training_dataset_sha256']
    == EXPECTED_TRAINING_HASH
)

assert (
    lr_char_config['validation_dataset_sha256']
    == EXPECTED_VALIDATION_HASH
)

assert not lr_char_config['official_test_used']

lr_char_summary = pd.read_csv(
    lr_char_run_dir / 'tables' / 'run_summary.csv'
).iloc[0]

assert np.isclose(
    lr_char_summary['Macro AP'],
    lr_char_record['validation_macro_ap'],
    rtol=0,
    atol=1e-6
)


# Compare LR and SVM using identical character features

lr_svm_char_comparison_df = pd.DataFrame([
    {
        'Metric': metric,
        'Logistic Regression': lr_char_summary[metric],
        'Linear SVM': char_result['summary'][metric],
        'Difference': (
            char_result['summary'][metric]
            - lr_char_summary[metric]
        )
    }
    for metric in comparison_metrics
])

print("\nLogistic Regression vs Linear SVM using character features")

display(lr_svm_char_comparison_df.round(6))

lr_svm_char_comparison_df.to_csv(
    TABLES_DIR / 'lr_vs_svm_character_comparison.csv',
    index=False
)


# Compare LR and SVM performance for each label

lr_char_per_label = pd.read_csv(
    lr_char_run_dir
    / 'tables'
    / 'optimized_per_label_metrics.csv'
)

assert lr_char_per_label['Label'].tolist() == LABELS

assert np.array_equal(
    lr_char_per_label['Positive Support'],
    char_metrics_df['Positive Support']
)

lr_svm_char_per_label_df = pd.DataFrame({
    'Label': LABELS,
    'LR Average Precision':
        lr_char_per_label['Average Precision'],
    'SVM Average Precision':
        char_metrics_df['Average Precision'],
    'LR Optimized F1':
        lr_char_per_label['F1'],
    'SVM Optimized F1':
        char_metrics_df['F1']
})

lr_svm_char_per_label_df['AP Difference'] = (
    lr_svm_char_per_label_df['SVM Average Precision']
    - lr_svm_char_per_label_df['LR Average Precision']
)

lr_svm_char_per_label_df['F1 Difference'] = (
    lr_svm_char_per_label_df['SVM Optimized F1']
    - lr_svm_char_per_label_df['LR Optimized F1']
)

print("\nPer-label LR vs SVM character-feature comparison")

display(lr_svm_char_per_label_df.round(6))

lr_svm_char_per_label_df.to_csv(
    TABLES_DIR / 'lr_vs_svm_character_per_label.csv',
    index=False
)


# Summarize the character representation experiment

char_summary = char_result['summary']
word_summary = word_result['summary']

print("\nCharacter TF-IDF experiment summary")

print("Macro AP:",
      round(char_summary['Macro AP'], 6))

print("Macro AP difference vs word:",
      round(
          char_summary['Macro AP']
          - word_summary['Macro AP'], 6
      ))

print("Default Macro F1:",
      round(char_summary['Default Macro F1'], 6))

print("Optimized Macro F1:",
      round(char_summary['Optimized Macro F1'], 6))

print("Optimized Macro F1 difference vs word:",
      round(
          char_summary['Optimized Macro F1']
          - word_summary['Optimized Macro F1'], 6
      ))

print("Labels with improved AP:",
      int(
          (word_char_per_label_df['AP Difference'] > 0).sum()
      ),
      "/", len(LABELS))

print("Labels with improved F1:",
      int(
          (word_char_per_label_df['F1 Difference'] > 0).sum()
      ),
      "/", len(LABELS))

print("Macro AP difference vs character LR:",
      round(
          char_summary['Macro AP']
          - lr_char_summary['Macro AP'], 6
      ))

print("Optimized Macro F1 difference vs character LR:",
      round(
          char_summary['Optimized Macro F1']
          - lr_char_summary['Optimized Macro F1'], 6
      ))

print("Exact zero decision scores:",
      char_summary['Exact Zero Decision Scores'])

print("Convergence warnings:",
      char_summary['Convergence Warnings'])

print("Completed SVM experiments:", len(svm_results))
print("Official test dataset: Not loaded")

print("\nCharacter-based Linear SVM experiment completed successfully.")

In [ ]:
# Train and evaluate the unweighted combined-feature Linear SVM

combined_spec = experiment_specs[2]

assert combined_spec['representation'] == 'combined_tfidf'
assert combined_spec['class_weight'] is None

combined_result = run_svm_experiment(combined_spec)

svm_results[combined_spec['run_id']] = combined_result


# Consolidate all three unweighted representation experiments

representation_results = [
    word_result,
    char_result,
    combined_result
]

representation_comparison_df = pd.DataFrame([
    result['summary']
    for result in representation_results
])

assert len(representation_comparison_df) == 3

assert representation_comparison_df[
    'Training Samples'
].nunique() == 1

assert representation_comparison_df[
    'Validation Samples'
].nunique() == 1

assert representation_comparison_df[
    'Features'
].nunique() == 1

assert representation_comparison_df[
    'Class Weight'
].eq('None').all()

assert representation_comparison_df[
    'Convergence Warnings'
].eq(0).all()

assert not representation_comparison_df[
    'Official Test Used'
].any()

comparison_columns = [
    'Feature Representation',
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Default Macro F1',
    'Optimized Macro F1',
    'Default Micro F1',
    'Optimized Micro F1',
    'Optimized Macro Precision',
    'Optimized Macro Recall',
    'Training Time (s)',
    'Validation Prediction Time (s)',
    'Model Size (MiB)'
]

print("\nOverall SVM representation comparison")

display(
    representation_comparison_df[
        comparison_columns
    ].round(6)
)

representation_comparison_df.to_csv(
    TABLES_DIR / 'svm_representation_comparison.csv',
    index=False
)


# Consolidate per-label metrics across representations

per_label_records = []

for result in representation_results:

    metrics_df = result['optimized_per_label'].copy()

    assert metrics_df['Label'].tolist() == LABELS

    assert np.array_equal(
        metrics_df['Positive Support'].to_numpy(),
        y_val.sum(axis=0)
    )

    metrics_df.insert(
        0,
        'Representation',
        result['spec']['representation']
    )

    per_label_records.append(metrics_df)

representation_per_label_df = pd.concat(
    per_label_records,
    ignore_index=True
)

representation_per_label_df.to_csv(
    TABLES_DIR / 'svm_representation_per_label_metrics.csv',
    index=False
)


# Create compact per-label AP and F1 tables

ap_comparison_df = (
    representation_per_label_df.pivot(
        index='Label',
        columns='Representation',
        values='Average Precision'
    )
    .reindex(
        index=LABELS,
        columns=REPRESENTATIONS
    )
)

f1_comparison_df = (
    representation_per_label_df.pivot(
        index='Label',
        columns='Representation',
        values='F1'
    )
    .reindex(
        index=LABELS,
        columns=REPRESENTATIONS
    )
)

assert not ap_comparison_df.isna().any().any()
assert not f1_comparison_df.isna().any().any()

print("\nPer-label Average Precision comparison")

display(ap_comparison_df.round(6))

print("\nPer-label optimized F1 comparison")

display(f1_comparison_df.round(6))

ap_comparison_df.to_csv(
    TABLES_DIR / 'svm_representation_ap_comparison.csv'
)

f1_comparison_df.to_csv(
    TABLES_DIR / 'svm_representation_f1_comparison.csv'
)


# Measure the effect of combining word and character features

combined_comparison_df = pd.DataFrame({
    'Label': LABELS,
    'Combined AP':
        ap_comparison_df['combined_tfidf'].to_numpy(),
    'Combined AP vs Word': (
        ap_comparison_df['combined_tfidf']
        - ap_comparison_df['word_tfidf']
    ).to_numpy(),
    'Combined AP vs Character': (
        ap_comparison_df['combined_tfidf']
        - ap_comparison_df['character_tfidf']
    ).to_numpy(),
    'Combined F1':
        f1_comparison_df['combined_tfidf'].to_numpy(),
    'Combined F1 vs Word': (
        f1_comparison_df['combined_tfidf']
        - f1_comparison_df['word_tfidf']
    ).to_numpy(),
    'Combined F1 vs Character': (
        f1_comparison_df['combined_tfidf']
        - f1_comparison_df['character_tfidf']
    ).to_numpy()
})

print("\nCombined representation gains and losses")

display(combined_comparison_df.round(6))

combined_comparison_df.to_csv(
    TABLES_DIR / 'svm_combined_representation_effect.csv',
    index=False
)


# Load the matched unweighted Logistic Regression experiment

lr_combined_record = registry_df.loc[
    registry_df['run_id'] == 'LR_COMBINED_BASELINE_S42'
].iloc[0]

lr_combined_run_dir = (
    PROJECT_ROOT / lr_combined_record['results_path']
)

lr_combined_config = previous_run_configs[
    'LR_COMBINED_BASELINE_S42'
]

assert (
    lr_combined_config['feature_representation']
    == 'combined_tfidf'
)

assert (
    lr_combined_config['model_parameters']['class_weight']
    is None
)

assert (
    lr_combined_config['training_dataset_sha256']
    == EXPECTED_TRAINING_HASH
)

assert (
    lr_combined_config['validation_dataset_sha256']
    == EXPECTED_VALIDATION_HASH
)

assert not lr_combined_config['official_test_used']

lr_combined_summary = pd.read_csv(
    lr_combined_run_dir / 'tables' / 'run_summary.csv'
).iloc[0]

assert np.isclose(
    lr_combined_summary['Macro AP'],
    lr_combined_record['validation_macro_ap'],
    atol=1e-6,
    rtol=0
)


# Compare Logistic Regression and SVM with combined features

lr_svm_combined_comparison_df = pd.DataFrame([
    {
        'Metric': metric,
        'Logistic Regression':
            lr_combined_summary[metric],
        'Linear SVM':
            combined_result['summary'][metric],
        'Difference': (
            combined_result['summary'][metric]
            - lr_combined_summary[metric]
        )
    }
    for metric in comparison_metrics
])

print("\nLR vs SVM using identical combined features")

display(lr_svm_combined_comparison_df.round(6))

lr_svm_combined_comparison_df.to_csv(
    TABLES_DIR / 'lr_vs_svm_combined_comparison.csv',
    index=False
)


# Compare both learning algorithms for each toxicity label

lr_combined_per_label = pd.read_csv(
    lr_combined_run_dir
    / 'tables'
    / 'optimized_per_label_metrics.csv'
)

svm_combined_per_label = combined_result[
    'optimized_per_label'
]

assert lr_combined_per_label['Label'].tolist() == LABELS

assert np.array_equal(
    lr_combined_per_label['Positive Support'],
    svm_combined_per_label['Positive Support']
)

lr_svm_combined_per_label_df = pd.DataFrame({
    'Label': LABELS,
    'LR Average Precision':
        lr_combined_per_label['Average Precision'],
    'SVM Average Precision':
        svm_combined_per_label['Average Precision'],
    'LR Optimized F1':
        lr_combined_per_label['F1'],
    'SVM Optimized F1':
        svm_combined_per_label['F1']
})

lr_svm_combined_per_label_df['AP Difference'] = (
    lr_svm_combined_per_label_df['SVM Average Precision']
    - lr_svm_combined_per_label_df['LR Average Precision']
)

lr_svm_combined_per_label_df['F1 Difference'] = (
    lr_svm_combined_per_label_df['SVM Optimized F1']
    - lr_svm_combined_per_label_df['LR Optimized F1']
)

print("\nPer-label LR vs SVM combined-feature comparison")

display(lr_svm_combined_per_label_df.round(6))

lr_svm_combined_per_label_df.to_csv(
    TABLES_DIR / 'lr_vs_svm_combined_per_label.csv',
    index=False
)


# Rank unweighted SVM configurations using validation Macro AP

representation_ranking_df = (
    representation_comparison_df[[
        'Run ID',
        'Feature Representation',
        'Macro AP',
        'Macro ROC-AUC',
        'Optimized Macro F1',
        'Optimized Micro F1',
        'Training Time (s)'
    ]]
    .sort_values(
        ['Macro AP', 'Optimized Macro F1'],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

representation_ranking_df.insert(
    0,
    'Rank',
    np.arange(
        1,
        len(representation_ranking_df) + 1
    )
)

print("\nUnweighted SVM representation ranking")

display(representation_ranking_df.round(6))

representation_ranking_df.to_csv(
    TABLES_DIR / 'svm_representation_ranking.csv',
    index=False
)

best_unweighted_run_id = representation_ranking_df.loc[
    0, 'Run ID'
]

best_unweighted_result = svm_results[
    best_unweighted_run_id
]

best_representation = best_unweighted_result[
    'spec'
]['representation']

combined_summary = combined_result['summary']

best_summary = best_unweighted_result['summary']


# Summarize the representation comparison

print("\nCombined TF-IDF experiment summary")

print("Combined Macro AP:",
      round(combined_summary['Macro AP'], 6))

print("Macro AP difference vs word:",
      round(
          combined_summary['Macro AP']
          - word_result['summary']['Macro AP'],
          6
      ))

print("Macro AP difference vs character:",
      round(
          combined_summary['Macro AP']
          - char_result['summary']['Macro AP'],
          6
      ))

print("Macro AP difference vs combined LR:",
      round(
          combined_summary['Macro AP']
          - lr_combined_summary['Macro AP'],
          6
      ))

print("Optimized Macro F1:",
      round(
          combined_summary['Optimized Macro F1'],
          6
      ))

print("\nRepresentation selection summary")

print("Best unweighted SVM run:",
      best_unweighted_run_id)

print("Selected representation:",
      best_representation)

print("Best validation Macro AP:",
      round(best_summary['Macro AP'], 6))

print("Best optimized Macro F1:",
      round(best_summary['Optimized Macro F1'], 6))

print("Selection metric: Macro AP")

print("Exact zero decision scores:",
      combined_summary['Exact Zero Decision Scores'])

print("Convergence warnings:",
      combined_summary['Convergence Warnings'])

print("Completed SVM experiments:", len(svm_results))
print("Official test dataset: Not loaded")

print("\nSVM representation comparison completed successfully.")

In [ ]:
# Prepare representation comparison data

from sklearn.metrics import precision_recall_curve

representation_names = {
    'word_tfidf': 'Word',
    'character_tfidf': 'Character',
    'combined_tfidf': 'Combined'
}

label_names = [
    label.replace('_', ' ').title()
    for label in LABELS
]

plot_colors = [
    REPRESENTATION_COLORS[representation]
    for representation in REPRESENTATIONS
]

plot_df = representation_comparison_df.set_index(
    'Feature Representation'
).loc[REPRESENTATIONS].copy()

plot_df.index = [
    representation_names[representation]
    for representation in plot_df.index
]


# Compare overall SVM representation performance

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, columns, title in [
    (
        axes[0],
        ['Macro AP', 'Macro ROC-AUC'],
        'Ranking Performance'
    ),
    (
        axes[1],
        [
            'Default Macro F1',
            'Optimized Macro F1',
            'Optimized Micro F1'
        ],
        'Classification Performance'
    )
]:

    plot_df[columns].T.plot(
        kind='bar',
        color=plot_colors,
        width=0.75,
        ax=ax
    )

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt='%.3f',
            padding=3,
            fontsize=9
        )

    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='x', visible=False)
    ax.legend(title='Representation')

fig.suptitle(
    'Linear SVM: TF-IDF Representation Comparison',
    fontsize=16
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.94])

fig.savefig(
    FIGURES_DIR / 'svm_representation_overall_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Compare per-label Average Precision and optimized F1

fig, axes = plt.subplots(2, 1, figsize=(15, 11))

for ax, metrics_df, title in [
    (
        axes[0],
        ap_comparison_df,
        'Per-Label Average Precision'
    ),
    (
        axes[1],
        f1_comparison_df,
        'Per-Label Optimized F1'
    )
]:

    chart_df = metrics_df.copy()

    chart_df.index = label_names

    chart_df.columns = [
        representation_names[column]
        for column in chart_df.columns
    ]

    chart_df.plot(
        kind='bar',
        color=plot_colors,
        width=0.75,
        ax=ax
    )

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt='%.3f',
            padding=3,
            fontsize=8
        )

    ax.set_title(title)
    ax.set_ylabel('Score')
    ax.set_xlabel('')
    ax.set_ylim(0, 1.08)
    ax.tick_params(axis='x', rotation=0)
    ax.grid(axis='x', visible=False)
    ax.legend(title='Representation')

fig.suptitle(
    'Linear SVM: Per-Label Representation Performance',
    fontsize=16
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.95])

fig.savefig(
    FIGURES_DIR / 'svm_representation_per_label_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Reload verified validation decision scores

representation_scores = {}

for result in representation_results:

    representation = result['spec']['representation']

    predictions_df = pd.read_parquet(
        result['predictions_path']
    )

    assert (
        predictions_df['id'].tolist()
        == val_df['id'].tolist()
    )

    saved_targets = predictions_df[[
        f'{label}_true' for label in LABELS
    ]].to_numpy()

    assert np.array_equal(saved_targets, y_val)

    scores = predictions_df[[
        f'{label}_score' for label in LABELS
    ]].to_numpy(dtype=np.float64)

    assert scores.shape == y_val.shape
    assert np.isfinite(scores).all()

    representation_scores[representation] = scores


# Generate precision-recall curves using raw decision scores

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

pr_curve_records = []

for i, (ax, label) in enumerate(zip(axes.flat, LABELS)):

    for representation in REPRESENTATIONS:

        precision, recall, thresholds = precision_recall_curve(
            y_val[:, i],
            representation_scores[representation][:, i]
        )

        ap = ap_comparison_df.loc[
            label, representation
        ]

        ax.step(
            recall,
            precision,
            where='post',
            linewidth=1.8,
            color=REPRESENTATION_COLORS[representation],
            label=(
                f'{representation_names[representation]} '
                f'(AP={ap:.4f})'
            )
        )

        pr_curve_records.append(pd.DataFrame({
            'Label': label,
            'Representation': representation,
            'Precision': precision,
            'Recall': recall,
            'Decision Threshold': np.append(
                thresholds, np.nan
            )
        }))

    ax.axhline(
        y_val[:, i].mean(),
        color='gray',
        linestyle='--',
        linewidth=1.2,
        label='Prevalence reference'
    )

    ax.set_title(label_names[i])
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

fig.suptitle(
    'Linear SVM: Representation Precision-Recall Curves',
    fontsize=16
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.95])

fig.savefig(
    FIGURES_DIR / 'svm_representation_precision_recall_curves.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

# Save the numerical data underlying the precision-recall curves

pr_curve_df = pd.concat(
    pr_curve_records,
    ignore_index=True
)

pr_curve_df.to_csv(
    TABLES_DIR / 'svm_representation_pr_curve_data.csv',
    index=False
)

del representation_scores
del predictions_df, saved_targets, scores
del pr_curve_records, pr_curve_df

gc.collect()


# Consolidate matched Logistic Regression and SVM comparisons

matched_comparison_records = []

lr_run_ids = {
    'word_tfidf': 'LR_WORD_BASELINE_S42',
    'character_tfidf': 'LR_CHAR_BASELINE_S42',
    'combined_tfidf': 'LR_COMBINED_BASELINE_S42'
}

matched_metrics = [
    'Macro AP',
    'Macro ROC-AUC',
    'Default Macro F1',
    'Optimized Macro F1',
    'Optimized Micro F1',
    'Training Time (s)',
    'Validation Prediction Time (s)',
    'Model Size (MiB)'
]

for result in representation_results:

    representation = result['spec']['representation']

    lr_run_id = lr_run_ids[representation]

    lr_record = registry_df.loc[
        registry_df['run_id'] == lr_run_id
    ].iloc[0]

    lr_config = previous_run_configs[lr_run_id]

    assert lr_config['feature_representation'] == representation
    assert lr_config['model_parameters']['class_weight'] is None
    assert lr_config['training_dataset_sha256'] == EXPECTED_TRAINING_HASH
    assert lr_config['validation_dataset_sha256'] == EXPECTED_VALIDATION_HASH
    assert not lr_config['official_test_used']

    lr_summary = pd.read_csv(
        PROJECT_ROOT
        / lr_record['results_path']
        / 'tables'
        / 'run_summary.csv'
    ).iloc[0]

    svm_summary = result['summary']

    for model_name, summary_record, run_id in [
        ('Logistic Regression', lr_summary, lr_run_id),
        ('Linear SVM', svm_summary, result['spec']['run_id'])
    ]:

        matched_comparison_records.append({
            'Run ID': run_id,
            'Model': model_name,
            'Representation': representation,
            **{
                metric: summary_record[metric]
                for metric in matched_metrics
            }
        })

matched_comparison_df = pd.DataFrame(
    matched_comparison_records
)

matched_comparison_df.to_csv(
    TABLES_DIR / 'lr_vs_svm_all_representations.csv',
    index=False
)

print("\nMatched Logistic Regression and Linear SVM comparison")

display(matched_comparison_df.round(6))


# Compare both algorithms across the three representations

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

algorithm_colors = ['#4C78A8', '#54A24B']

for ax, metric in zip(
    axes.flat,
    [
        'Macro AP',
        'Macro ROC-AUC',
        'Optimized Macro F1',
        'Training Time (s)'
    ]
):

    chart_df = matched_comparison_df.pivot(
        index='Representation',
        columns='Model',
        values=metric
    ).reindex(
        index=REPRESENTATIONS,
        columns=['Logistic Regression', 'Linear SVM']
    )

    chart_df.index = [
        representation_names[representation]
        for representation in chart_df.index
    ]

    chart_df.plot(
        kind='bar',
        ax=ax,
        color=algorithm_colors,
        width=0.7
    )

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt='%.3f',
            padding=3,
            fontsize=9
        )

    ax.set_title(metric)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
    ax.grid(axis='x', visible=False)

    if metric == 'Training Time (s)':

        ax.set_ylabel('Seconds')
        ax.set_ylim(
            0,
            chart_df.to_numpy().max() * 1.2
        )

    else:

        ax.set_ylabel('Score')
        ax.set_ylim(0, 1.1)

    ax.legend(fontsize=9)

fig.suptitle(
    'Logistic Regression vs Linear SVM: Matched Representations',
    fontsize=16
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.95])

fig.savefig(
    FIGURES_DIR / 'lr_vs_svm_matched_representation_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Record computational efficiency across SVM representations

efficiency_records = []

for result in representation_results:

    representation = result['spec']['representation']

    feature_rows = feature_verification_df.loc[
        feature_verification_df['Representation']
        == representation
    ]

    efficiency_records.append({
        'Representation': representation,
        'Training Time (s)':
            result['summary']['Training Time (s)'],
        'Validation Prediction Time (s)':
            result['summary']['Validation Prediction Time (s)'],
        'Training Matrix Memory (MiB)': feature_rows.loc[
            feature_rows['Partition'] == 'train',
            'Sparse Memory (MiB)'
        ].iloc[0],
        'Validation Matrix Memory (MiB)': feature_rows.loc[
            feature_rows['Partition'] == 'validation',
            'Sparse Memory (MiB)'
        ].iloc[0],
        'Model Size (MiB)':
            result['summary']['Model Size (MiB)']
    })

efficiency_df = pd.DataFrame(efficiency_records)

efficiency_df.to_csv(
    TABLES_DIR / 'svm_representation_efficiency.csv',
    index=False
)

print("\nComputational efficiency comparison")

display(efficiency_df.round(6))


# Document representation selection before class weighting

best_summary = best_unweighted_result['summary']

runner_up = representation_ranking_df.iloc[1]

selection_config = {
    'model': MODEL_NAME,
    'selection_partition': 'validation',
    'selection_metric': 'macro_average_precision',
    'secondary_metric': 'optimized_macro_f1',
    'selected_run_id': best_unweighted_run_id,
    'selected_representation': best_representation,
    'selected_macro_ap': float(
        best_summary['Macro AP']
    ),
    'selected_optimized_macro_f1': float(
        best_summary['Optimized Macro F1']
    ),
    'macro_ap_gain_over_word': float(
        best_summary['Macro AP']
        - word_result['summary']['Macro AP']
    ),
    'macro_ap_gain_over_runner_up': float(
        best_summary['Macro AP']
        - runner_up['Macro AP']
    ),
    'feature_budget': FEATURE_BUDGET,
    'score_type': SCORE_TYPE,
    'statistical_significance_established': False,
    'next_experiment': 'balanced_class_weight',
    'official_test_used': False
}

SVM_REPRESENTATION_SELECTION_PATH = (
    RUN_CONFIGS_DIR / 'svm_representation_selection.json'
)

with open(
    SVM_REPRESENTATION_SELECTION_PATH, 'w'
) as file:

    json.dump(selection_config, file, indent=4)

with open(
    SVM_REPRESENTATION_SELECTION_PATH, 'r'
) as file:

    saved_selection = json.load(file)

assert saved_selection == selection_config


# Summarize the completed representation comparison

print("\nRepresentation selection summary")

print("Selected representation:", best_representation)

print("Selected run:", best_unweighted_run_id)

print("Validation Macro AP:",
      round(best_summary['Macro AP'], 6))

print("Optimized Macro F1:",
      round(best_summary['Optimized Macro F1'], 6))

print("Macro AP gain over word:",
      round(
          selection_config['macro_ap_gain_over_word'],
          6
      ))

print("Macro AP gain over runner-up:",
      round(
          selection_config['macro_ap_gain_over_runner_up'],
          6
      ))

print("Research figures generated: 4")
print("Matched LR-SVM configurations compared: 6")
print("Representation selection: Saved and verified")
print("Next experiment: Balanced class weighting")
print("Official test dataset: Not used")

print("\nSVM representation visualization completed successfully.")

In [ ]:
# Train balanced Linear SVM using the selected representation

with open(SVM_REPRESENTATION_SELECTION_PATH, 'r') as file:
    saved_selection = json.load(file)

assert saved_selection['selected_representation'] == best_representation
assert saved_selection['selected_run_id'] == best_unweighted_run_id
assert not saved_selection['official_test_used']

assert best_representation == 'combined_tfidf'
assert best_unweighted_result['spec']['class_weight'] is None

weighted_spec = {
    'run_id': 'SVM_COMBINED_BALANCED_S42',
    'representation': best_representation,
    'class_weight': 'balanced',
    'purpose': 'Controlled class imbalance handling experiment'
}

weighted_result = run_svm_experiment(weighted_spec)

svm_results[weighted_spec['run_id']] = weighted_result


# Compare overall weighted and unweighted performance

unweighted_summary = best_unweighted_result['summary']
weighted_summary = weighted_result['summary']

comparison_metrics = [
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Default Macro F1',
    'Optimized Macro F1',
    'Default Micro F1',
    'Optimized Micro F1',
    'Optimized Macro Precision',
    'Optimized Macro Recall',
    'Training Time (s)',
    'Validation Prediction Time (s)',
    'Model Size (MiB)'
]

weighting_comparison_df = pd.DataFrame([
    {
        'Metric': metric,
        'Unweighted': unweighted_summary[metric],
        'Balanced': weighted_summary[metric],
        'Change': (
            weighted_summary[metric]
            - unweighted_summary[metric]
        )
    }
    for metric in comparison_metrics
])

print("\nUnweighted vs Balanced Linear SVM")

display(weighting_comparison_df.round(6))

weighting_comparison_df.to_csv(
    TABLES_DIR / 'svm_class_weighting_comparison.csv',
    index=False
)


# Compare individual toxicity labels at optimized thresholds

unweighted_per_label = (
    best_unweighted_result['optimized_per_label']
)

weighted_per_label = weighted_result[
    'optimized_per_label'
]

assert unweighted_per_label['Label'].tolist() == LABELS
assert weighted_per_label['Label'].tolist() == LABELS

assert np.array_equal(
    unweighted_per_label['Positive Support'],
    weighted_per_label['Positive Support']
)

assert np.array_equal(
    unweighted_per_label['Negative Support'],
    weighted_per_label['Negative Support']
)

weighting_per_label_df = pd.DataFrame({
    'Label': LABELS,
    'Actual Positives':
        unweighted_per_label['Positive Support'],
    'Unweighted AP':
        unweighted_per_label['Average Precision'],
    'Balanced AP':
        weighted_per_label['Average Precision'],
    'Unweighted Precision':
        unweighted_per_label['Precision'],
    'Balanced Precision':
        weighted_per_label['Precision'],
    'Unweighted Recall':
        unweighted_per_label['Recall'],
    'Balanced Recall':
        weighted_per_label['Recall'],
    'Unweighted F1':
        unweighted_per_label['F1'],
    'Balanced F1':
        weighted_per_label['F1']
})

for metric in ['AP', 'Precision', 'Recall', 'F1']:

    weighting_per_label_df[f'{metric} Change'] = (
        weighting_per_label_df[f'Balanced {metric}']
        - weighting_per_label_df[f'Unweighted {metric}']
    )

print("\nPer-label class weighting comparison")

display(weighting_per_label_df.round(6))

weighting_per_label_df.to_csv(
    TABLES_DIR / 'svm_class_weighting_per_label.csv',
    index=False
)


# Compare classification errors at optimized thresholds

weighting_errors_df = pd.DataFrame({
    'Label': LABELS,
    'Unweighted TP': unweighted_per_label['TP'],
    'Balanced TP': weighted_per_label['TP'],
    'Unweighted FP': unweighted_per_label['FP'],
    'Balanced FP': weighted_per_label['FP'],
    'Unweighted FN': unweighted_per_label['FN'],
    'Balanced FN': weighted_per_label['FN'],
    'Unweighted TN': unweighted_per_label['TN'],
    'Balanced TN': weighted_per_label['TN']
})

for error_type in ['TP', 'FP', 'FN', 'TN']:

    weighting_errors_df[f'{error_type} Change'] = (
        weighting_errors_df[f'Balanced {error_type}']
        - weighting_errors_df[f'Unweighted {error_type}']
    )

assert np.array_equal(
    weighting_errors_df['Unweighted TP']
    + weighting_errors_df['Unweighted FN'],
    y_val.sum(axis=0)
)

assert np.array_equal(
    weighting_errors_df['Balanced TP']
    + weighting_errors_df['Balanced FN'],
    y_val.sum(axis=0)
)

print("\nClassification error comparison")

display(weighting_errors_df)

weighting_errors_df.to_csv(
    TABLES_DIR / 'svm_class_weighting_error_comparison.csv',
    index=False
)


# Compare optimized signed decision thresholds

weighting_threshold_df = pd.DataFrame({
    'Label': LABELS,
    'Default Threshold': DEFAULT_THRESHOLD,
    'Unweighted Threshold':
        unweighted_per_label['Threshold'],
    'Balanced Threshold':
        weighted_per_label['Threshold']
})

weighting_threshold_df['Threshold Change'] = (
    weighting_threshold_df['Balanced Threshold']
    - weighting_threshold_df['Unweighted Threshold']
)

print("\nOptimized decision threshold comparison")

display(weighting_threshold_df.round(8))

weighting_threshold_df.to_csv(
    TABLES_DIR / 'svm_class_weighting_threshold_comparison.csv',
    index=False
)


# Compare rare-label performance separately

RARE_LABELS = [
    'severe_toxic',
    'threat',
    'identity_hate'
]

rare_label_comparison_df = weighting_per_label_df.loc[
    weighting_per_label_df['Label'].isin(RARE_LABELS)
].copy()

assert len(rare_label_comparison_df) == 3

print("\nRare-label performance comparison")

display(rare_label_comparison_df.round(6))

rare_label_comparison_df.to_csv(
    TABLES_DIR / 'svm_class_weighting_rare_labels.csv',
    index=False
)


# Compare both models before and after threshold optimization

threshold_effect_records = []

for name, result in [
    ('Unweighted', best_unweighted_result),
    ('Balanced', weighted_result)
]:

    for threshold_type, metrics in [
        ('Default', result['default_metrics']),
        ('Optimized', result['optimized_metrics'])
    ]:

        threshold_effect_records.append({
            'Class Weight': name,
            'Threshold Type': threshold_type,
            'Macro AP': metrics['Macro AP'],
            'Macro Precision': metrics['Macro Precision'],
            'Macro Recall': metrics['Macro Recall'],
            'Macro F1': metrics['Macro F1'],
            'Micro F1': metrics['Micro F1'],
            'Hamming Loss': metrics['Hamming Loss'],
            'Exact Match Accuracy':
                metrics['Exact Match Accuracy']
        })

weighting_threshold_effect_df = pd.DataFrame(
    threshold_effect_records
)

print("\nJoint effect of class weighting and threshold optimization")

display(weighting_threshold_effect_df.round(6))

weighting_threshold_effect_df.to_csv(
    TABLES_DIR / 'svm_weighting_threshold_interaction.csv',
    index=False
)


# Compare class weighting effects across LR and SVM

lr_weighted_record = registry_df.loc[
    registry_df['run_id'] == 'LR_COMBINED_BALANCED_S42'
].iloc[0]

lr_weighted_config = previous_run_configs[
    'LR_COMBINED_BALANCED_S42'
]

assert lr_weighted_config['feature_representation'] == best_representation
assert lr_weighted_config['model_parameters']['class_weight'] == 'balanced'

assert (
    lr_weighted_config['training_dataset_sha256']
    == EXPECTED_TRAINING_HASH
)

assert (
    lr_weighted_config['validation_dataset_sha256']
    == EXPECTED_VALIDATION_HASH
)

assert not lr_weighted_config['official_test_used']

lr_weighted_run_dir = (
    PROJECT_ROOT / lr_weighted_record['results_path']
)

lr_weighted_summary = pd.read_csv(
    lr_weighted_run_dir / 'tables' / 'run_summary.csv'
).iloc[0]

assert np.isclose(
    lr_weighted_summary['Macro AP'],
    lr_weighted_record['validation_macro_ap'],
    rtol=0,
    atol=1e-6
)

algorithm_weighting_records = []

for algorithm, unweighted, weighted in [
    (
        'Logistic Regression',
        lr_combined_summary,
        lr_weighted_summary
    ),
    (
        'Linear SVM',
        unweighted_summary,
        weighted_summary
    )
]:

    for metric in [
        'Macro AP',
        'Macro ROC-AUC',
        'Default Macro F1',
        'Optimized Macro F1',
        'Optimized Micro F1',
        'Optimized Macro Precision',
        'Optimized Macro Recall',
        'Training Time (s)'
    ]:

        algorithm_weighting_records.append({
            'Model': algorithm,
            'Metric': metric,
            'Unweighted': unweighted[metric],
            'Balanced': weighted[metric],
            'Change': (
                weighted[metric]
                - unweighted[metric]
            )
        })

algorithm_weighting_df = pd.DataFrame(
    algorithm_weighting_records
)

print("\nClass weighting effects across LR and SVM")

display(algorithm_weighting_df.round(6))

algorithm_weighting_df.to_csv(
    TABLES_DIR / 'lr_vs_svm_class_weighting_effect.csv',
    index=False
)


# Compare the two balanced models for each toxicity label

lr_weighted_per_label = pd.read_csv(
    lr_weighted_run_dir
    / 'tables'
    / 'optimized_per_label_metrics.csv'
)

assert lr_weighted_per_label['Label'].tolist() == LABELS

assert np.array_equal(
    lr_weighted_per_label['Positive Support'],
    weighted_per_label['Positive Support']
)

balanced_algorithm_comparison_df = pd.DataFrame({
    'Label': LABELS,
    'Actual Positives':
        weighted_per_label['Positive Support'],
    'Balanced LR AP':
        lr_weighted_per_label['Average Precision'],
    'Balanced SVM AP':
        weighted_per_label['Average Precision'],
    'Balanced LR F1':
        lr_weighted_per_label['F1'],
    'Balanced SVM F1':
        weighted_per_label['F1']
})

balanced_algorithm_comparison_df['AP Difference'] = (
    balanced_algorithm_comparison_df['Balanced SVM AP']
    - balanced_algorithm_comparison_df['Balanced LR AP']
)

balanced_algorithm_comparison_df['F1 Difference'] = (
    balanced_algorithm_comparison_df['Balanced SVM F1']
    - balanced_algorithm_comparison_df['Balanced LR F1']
)

print("\nBalanced LR vs Balanced SVM per-label comparison")

display(balanced_algorithm_comparison_df.round(6))

balanced_algorithm_comparison_df.to_csv(
    TABLES_DIR / 'lr_vs_svm_balanced_per_label.csv',
    index=False
)


# Summarize class weighting effectiveness

print("\nClass weighting experiment summary")

print("Unweighted Macro AP:",
      round(unweighted_summary['Macro AP'], 6))

print("Balanced Macro AP:",
      round(weighted_summary['Macro AP'], 6))

print("Macro AP change:",
      round(
          weighted_summary['Macro AP']
          - unweighted_summary['Macro AP'], 6
      ))

print("Unweighted optimized Macro F1:",
      round(unweighted_summary['Optimized Macro F1'], 6))

print("Balanced optimized Macro F1:",
      round(weighted_summary['Optimized Macro F1'], 6))

print("Optimized Macro F1 change:",
      round(
          weighted_summary['Optimized Macro F1']
          - unweighted_summary['Optimized Macro F1'], 6
      ))

print("Labels with improved AP:",
      int(
          (weighting_per_label_df['AP Change'] > 0).sum()
      ),
      "/", len(LABELS))

print("Labels with improved F1:",
      int(
          (weighting_per_label_df['F1 Change'] > 0).sum()
      ),
      "/", len(LABELS))

print("Unweighted rare-label mean AP:",
      round(
          rare_label_comparison_df['Unweighted AP'].mean(),
          6
      ))

print("Balanced rare-label mean AP:",
      round(
          rare_label_comparison_df['Balanced AP'].mean(),
          6
      ))

print("Unweighted rare-label mean F1:",
      round(
          rare_label_comparison_df['Unweighted F1'].mean(),
          6
      ))

print("Balanced rare-label mean F1:",
      round(
          rare_label_comparison_df['Balanced F1'].mean(),
          6
      ))

print("Balanced LR Macro AP:",
      round(lr_weighted_summary['Macro AP'], 6))

print("Balanced SVM Macro AP:",
      round(weighted_summary['Macro AP'], 6))

print("Balanced SVM AP difference vs Balanced LR:",
      round(
          weighted_summary['Macro AP']
          - lr_weighted_summary['Macro AP'], 6
      ))

print("Exact zero decision scores:",
      weighted_summary['Exact Zero Decision Scores'])

print("Convergence warnings:",
      weighted_summary['Convergence Warnings'])

print("Completed SVM experiments:", len(svm_results))
print("Official test dataset: Not loaded")

print("\nLinear SVM class weighting experiment completed successfully.")

In [ ]:
# Visualize per-label effects of balanced class weighting

weighting_colors = ['#4C78A8', '#54A24B']

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

for ax, metric in zip(
    axes.flat,
    ['AP', 'Precision', 'Recall', 'F1']
):

    chart_df = weighting_per_label_df.set_index('Label')[[
        f'Unweighted {metric}',
        f'Balanced {metric}'
    ]].copy()

    chart_df.index = label_names
    chart_df.columns = ['Unweighted', 'Balanced']

    chart_df.plot(
        kind='bar',
        ax=ax,
        color=weighting_colors,
        width=0.7
    )

    for container in ax.containers:
        ax.bar_label(
            container,
            fmt='%.3f',
            padding=3,
            fontsize=9
        )

    title = (
        'Average Precision'
        if metric == 'AP'
        else f'Optimized {metric}'
    )

    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='x', visible=False)
    ax.legend()

fig.suptitle(
    'Linear SVM: Effect of Balanced Class Weighting',
    fontsize=16
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.95])

fig.savefig(
    FIGURES_DIR / 'svm_class_weighting_per_label_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Compare class weighting effects across both linear algorithms

effect_metrics = [
    'Macro AP',
    'Default Macro F1',
    'Optimized Macro F1'
]

effect_plot_df = (
    algorithm_weighting_df.pivot(
        index='Metric',
        columns='Model',
        values='Change'
    )
    .reindex(
        index=effect_metrics,
        columns=['Logistic Regression', 'Linear SVM']
    )
)

effect_plot_df = effect_plot_df * 100

fig, ax = plt.subplots(figsize=(11, 6))

effect_plot_df.plot(
    kind='bar',
    ax=ax,
    color=['#4C78A8', '#54A24B'],
    width=0.65
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%+.2f',
        padding=5,
        fontsize=10
    )

ax.axhline(
    0,
    color='black',
    linewidth=1
)

ax.set_title(
    'Effect of Balanced Class Weighting: Logistic Regression vs Linear SVM',
    fontsize=14
)

ax.set_ylabel('Change in score (percentage points)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='x', visible=False)

minimum_change = effect_plot_df.to_numpy().min()
maximum_change = effect_plot_df.to_numpy().max()

ax.set_ylim(
    min(-2, minimum_change - 2),
    maximum_change + 2
)

ax.legend(title='Model')

sns.despine()
fig.tight_layout()

fig.savefig(
    FIGURES_DIR / 'lr_vs_svm_class_weighting_effect.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Consolidate all four completed SVM experiments

svm_comparison_df = pd.DataFrame([
    result['summary']
    for result in svm_results.values()
])

assert len(svm_comparison_df) == 4
assert svm_comparison_df['Run ID'].is_unique

assert svm_comparison_df[
    'Training Samples'
].nunique() == 1

assert svm_comparison_df[
    'Validation Samples'
].nunique() == 1

assert svm_comparison_df[
    'Features'
].nunique() == 1

assert svm_comparison_df[
    'Convergence Warnings'
].eq(0).all()

assert svm_comparison_df[
    'Status'
].eq('Completed').all()

assert not svm_comparison_df[
    'Official Test Used'
].any()

assert svm_comparison_df[
    'Maximum Score Reconstruction Error'
].le(1e-12).all()

svm_comparison_df.to_csv(
    TABLES_DIR / 'svm_all_experiments_comparison.csv',
    index=False
)

run_display_names = {
    'SVM_WORD_BASELINE_S42': 'Word',
    'SVM_CHAR_BASELINE_S42': 'Character',
    'SVM_COMBINED_BASELINE_S42': 'Combined',
    'SVM_COMBINED_BALANCED_S42': 'Combined Balanced'
}

svm_plot_df = svm_comparison_df.copy()

svm_plot_df['Configuration'] = svm_plot_df[
    'Run ID'
].map(run_display_names)

assert svm_plot_df['Configuration'].notna().all()

svm_plot_df = svm_plot_df.set_index(
    'Configuration'
)


# Compare all four SVM configurations

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

comparison_colors = [
    '#4C78A8',
    '#F58518',
    '#54A24B',
    '#B279A2'
]

for ax, metrics, title in [
    (
        axes[0],
        ['Macro AP', 'Macro ROC-AUC'],
        'Ranking Performance'
    ),
    (
        axes[1],
        [
            'Default Macro F1',
            'Optimized Macro F1',
            'Optimized Micro F1'
        ],
        'Classification Performance'
    )
]:

    svm_plot_df[metrics].T.plot(
        kind='bar',
        ax=ax,
        color=comparison_colors,
        width=0.8
    )

    for container in ax.containers:
        ax.bar_label(
            container,
            fmt='%.3f',
            padding=3,
            fontsize=8
        )

    ax.set_title(title)
    ax.set_ylabel('Score')
    ax.set_xlabel('')
    ax.set_ylim(0, 1.13)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='x', visible=False)
    ax.legend(fontsize=9)

fig.suptitle(
    'Linear SVM: Final Configuration Comparison',
    fontsize=16
)

sns.despine()
fig.tight_layout(rect=[0, 0, 1, 0.94])

fig.savefig(
    FIGURES_DIR / 'svm_final_configuration_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Rank configurations using the established selection criteria

svm_final_ranking_df = (
    svm_comparison_df[[
        'Run ID',
        'Feature Representation',
        'Class Weight',
        'Macro AP',
        'Macro ROC-AUC',
        'Default Macro F1',
        'Optimized Macro F1',
        'Optimized Micro F1',
        'Training Time (s)'
    ]]
    .sort_values(
        ['Macro AP', 'Optimized Macro F1', 'Run ID'],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

svm_final_ranking_df.insert(
    0,
    'Rank',
    np.arange(1, len(svm_final_ranking_df) + 1)
)

print("Final Linear SVM configuration ranking")

display(svm_final_ranking_df.round(6))

svm_final_ranking_df.to_csv(
    TABLES_DIR / 'svm_final_experiment_ranking.csv',
    index=False
)

best_svm_run_id = svm_final_ranking_df.loc[
    0, 'Run ID'
]

best_svm_result = svm_results[
    best_svm_run_id
]

best_svm_summary = best_svm_result['summary']


# Compare the selected LR and SVM configurations

lr_selection_path = (
    CLASSICAL_DIR / '02_logistic_regression'
    / 'results' / 'configs' / 'lr_final_selection.json'
)

with open(lr_selection_path, 'r') as file:
    lr_final_selection = json.load(file)

assert not lr_final_selection['official_test_used']
assert not lr_final_selection['final_project_model_frozen']

assert lr_final_selection['selected_run_id'] == (
    'LR_COMBINED_BALANCED_S42'
)

selected_linear_models_df = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'Run ID': lr_final_selection['selected_run_id'],
        'Representation':
            lr_final_selection['selected_representation'],
        'Class Weight':
            lr_final_selection['selected_class_weight'],
        'Macro AP':
            lr_final_selection['validation_macro_ap'],
        'Macro ROC-AUC':
            lr_final_selection['validation_macro_roc_auc'],
        'Default Macro F1':
            lr_final_selection['validation_default_macro_f1'],
        'Optimized Macro F1':
            lr_final_selection['validation_optimized_macro_f1'],
        'Optimized Micro F1':
            lr_final_selection['validation_optimized_micro_f1'],
        'Training Time (s)':
            lr_weighted_summary['Training Time (s)']
    },
    {
        'Model': MODEL_NAME,
        'Run ID': best_svm_run_id,
        'Representation':
            best_svm_result['spec']['representation'],
        'Class Weight':
            best_svm_summary['Class Weight'],
        'Macro AP':
            best_svm_summary['Macro AP'],
        'Macro ROC-AUC':
            best_svm_summary['Macro ROC-AUC'],
        'Default Macro F1':
            best_svm_summary['Default Macro F1'],
        'Optimized Macro F1':
            best_svm_summary['Optimized Macro F1'],
        'Optimized Micro F1':
            best_svm_summary['Optimized Micro F1'],
        'Training Time (s)':
            best_svm_summary['Training Time (s)']
    }
])

selected_linear_models_df = (
    selected_linear_models_df.sort_values(
        ['Macro AP', 'Optimized Macro F1'],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

print("\nSelected Logistic Regression vs Linear SVM")

display(selected_linear_models_df.round(6))

selected_linear_models_df.to_csv(
    TABLES_DIR / 'selected_lr_vs_svm_comparison.csv',
    index=False
)


# Record the class weighting decision using observed results

macro_ap_change = float(
    weighted_summary['Macro AP']
    - unweighted_summary['Macro AP']
)

macro_f1_change = float(
    weighted_summary['Optimized Macro F1']
    - unweighted_summary['Optimized Macro F1']
)

weighting_effect = {
    'unweighted_run_id': best_unweighted_run_id,
    'weighted_run_id': weighted_spec['run_id'],
    'macro_ap_change': macro_ap_change,
    'optimized_macro_f1_change': macro_f1_change,
    'default_macro_f1_change': float(
        weighted_summary['Default Macro F1']
        - unweighted_summary['Default Macro F1']
    ),
    'labels_with_improved_ap': int(
        (weighting_per_label_df['AP Change'] > 0).sum()
    ),
    'labels_with_improved_f1': int(
        (weighting_per_label_df['F1 Change'] > 0).sum()
    ),
    'training_time_ratio': float(
        weighted_summary['Training Time (s)']
        / unweighted_summary['Training Time (s)']
    ),
    'improved_primary_selection_metric':
        macro_ap_change > 0,
    'retained_for_research_comparison': True
}

with open(
    RUN_CONFIGS_DIR / 'svm_class_weighting_decision.json',
    'w'
) as file:
    json.dump(weighting_effect, file, indent=4)


# Save the selected SVM configuration

svm_final_selection = {
    'model': MODEL_NAME,
    'selection_partition': 'validation',
    'primary_selection_metric':
        'macro_average_precision',
    'secondary_selection_metric':
        'optimized_macro_f1',
    'remaining_tie_breaker':
        'ascending_run_id',
    'selected_run_id': best_svm_run_id,
    'selected_representation':
        best_svm_result['spec']['representation'],
    'selected_class_weight':
        best_svm_result['spec']['class_weight'],
    'score_type': SCORE_TYPE,
    'default_threshold': DEFAULT_THRESHOLD,
    'threshold_comparison_operator': '>=',
    'probability_calibration': False,
    'validation_macro_ap':
        float(best_svm_summary['Macro AP']),
    'validation_macro_roc_auc':
        float(best_svm_summary['Macro ROC-AUC']),
    'validation_default_macro_f1':
        float(best_svm_summary['Default Macro F1']),
    'validation_optimized_macro_f1':
        float(best_svm_summary['Optimized Macro F1']),
    'validation_optimized_micro_f1':
        float(best_svm_summary['Optimized Micro F1']),
    'model_path': str(
        best_svm_result['model_path'].relative_to(
            PROJECT_ROOT
        )
    ),
    'threshold_path': str(
        (
            RUN_CONFIGS_DIR
            / f'{best_svm_run_id}_thresholds.json'
        ).relative_to(PROJECT_ROOT)
    ),
    'predictions_path': str(
        best_svm_result['predictions_path'].relative_to(
            PROJECT_ROOT
        )
    ),
    'selection_scope':
        'Linear SVM configurations only',
    'statistical_significance_established': False,
    'final_project_model_frozen': False,
    'official_test_used': False
}

SVM_FINAL_SELECTION_PATH = (
    RUN_CONFIGS_DIR / 'svm_final_selection.json'
)

with open(SVM_FINAL_SELECTION_PATH, 'w') as file:
    json.dump(
        svm_final_selection,
        file,
        indent=4
    )

with open(SVM_FINAL_SELECTION_PATH, 'r') as file:
    saved_selection = json.load(file)

assert saved_selection == svm_final_selection

for field in [
    'model_path',
    'threshold_path',
    'predictions_path'
]:
    assert (
        PROJECT_ROOT / saved_selection[field]
    ).is_file()


# Summarize final SVM selection

print("\nLinear SVM selection summary")

print("Selected run:", best_svm_run_id)

print("Selected representation:",
      svm_final_selection['selected_representation'])

print("Selected class weight:",
      svm_final_selection['selected_class_weight'])

print("Validation Macro AP:",
      round(best_svm_summary['Macro AP'], 6))

print("Validation Macro ROC-AUC:",
      round(best_svm_summary['Macro ROC-AUC'], 6))

print("Optimized Macro F1:",
      round(best_svm_summary['Optimized Macro F1'], 6))

print("Optimized Micro F1:",
      round(best_svm_summary['Optimized Micro F1'], 6))

print("Class weighting improved Macro AP:",
      weighting_effect['improved_primary_selection_metric'])

print("Weighted-to-unweighted training time ratio:",
      round(weighting_effect['training_time_ratio'], 4))

print("Leading linear model:",
      selected_linear_models_df.loc[0, 'Model'])

print("Leading linear model run:",
      selected_linear_models_df.loc[0, 'Run ID'])

print("Completed SVM experiments:", len(svm_results))
print("Additional research figures generated: 3")
print("Selection configuration: Saved and verified")
print("Final project model frozen: No")
print("Official test dataset: Not used")

print("\nLinear SVM comparison completed successfully.")

In [ ]:
# Verify completed SVM experiments before registration

assert len(svm_results) == 4

registry_records = []
registration_checks = []

for run_id, result in svm_results.items():

    assert result['summary']['Status'] == 'Completed'
    assert result['summary']['Convergence Warnings'] == 0
    assert not result['summary']['Official Test Used']

    assert (
        result['summary']['Maximum Score Reconstruction Error']
        <= 1e-12
    )

    run_dir = result['results_path']

    manifest_df = pd.read_csv(
        run_dir / 'tables' / 'artifact_manifest.csv'
    )

    assert len(manifest_df) == 7
    assert manifest_df['Run ID'].eq(run_id).all()
    assert manifest_df['Artifact Type'].is_unique

    # Verify critical artifact fingerprints

    for _, artifact in manifest_df.iterrows():

        path = PROJECT_ROOT / artifact['Path']

        assert path.is_file()
        assert path.stat().st_size == artifact['Size Bytes']

        assert (
            calculate_sha256(path)
            == artifact['SHA256']
        ), f"Artifact fingerprint mismatch: {path}"

    # Verify the saved training log

    with open(
        run_dir / 'logs' / 'training_log.json',
        'r'
    ) as file:
        training_log = json.load(file)

    assert training_log['run_id'] == run_id
    assert training_log['status'] == 'Completed'
    assert training_log['all_classifiers_converged']
    assert training_log['model_reconstruction_verified']
    assert training_log['metric_reconstruction_verified']
    assert training_log['score_alignment_verified']
    assert training_log['threshold_roundtrip_verified']
    assert training_log['prediction_roundtrip_verified']
    assert not training_log['official_test_used']

    # Prepare the experiment registry entry

    record = result['registry_record'].copy()

    assert record['run_id'] == run_id
    assert record['status'] == 'Completed'

    assert (PROJECT_ROOT / record['model_path']).is_file()
    assert (PROJECT_ROOT / record['config_path']).is_file()
    assert (PROJECT_ROOT / record['results_path']).is_dir()

    if run_id == best_svm_run_id:

        record['notes'] += (
            ' Selected Linear SVM configuration '
            'based on validation Macro AP.'
        )

    if run_id == weighted_spec['run_id']:

        record['notes'] += (
            ' Balanced weighting improved default Macro F1 '
            'but reduced Macro AP and optimized Macro F1. '
            'Retained for controlled research comparison.'
        )

    registry_records.append(record)

    registration_checks.append({
        'Run ID': run_id,
        'Fingerprinted Artifacts Verified': len(manifest_df),
        'Convergence Verified': True,
        'Model Reconstruction Verified': True,
        'Metric Reconstruction Verified': True,
        'Decision Score Alignment Verified': True,
        'Exact Zero Decision Scores':
            result['summary']['Exact Zero Decision Scores'],
        'Official Test Used': False,
        'Selected SVM Configuration':
            run_id == best_svm_run_id,
        'Status': 'Verified'
    })

registration_verification_df = pd.DataFrame(
    registration_checks
)

print("Experiment registration verification")

display(registration_verification_df)

registration_verification_df.to_csv(
    TABLES_DIR / 'svm_registration_verification.csv',
    index=False
)


# Load the existing shared experiment registry

registry_before_df = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert registry_before_df['run_id'].is_unique

for record in registry_records:

    assert set(record.keys()) == set(
        registry_before_df.columns
    )

svm_run_ids = [
    record['run_id']
    for record in registry_records
]

# Preserve previously registered experiments

previous_registry_df = registry_before_df.loc[
    ~registry_before_df['run_id'].isin(svm_run_ids)
].copy()

assert set(required_previous_runs).issubset(
    set(previous_registry_df['run_id'])
)

new_registry_df = pd.DataFrame(
    registry_records,
    columns=registry_before_df.columns
).astype(str)

updated_registry_df = pd.concat(
    [previous_registry_df, new_registry_df],
    ignore_index=True
)

assert updated_registry_df['run_id'].is_unique

assert len(updated_registry_df) == (
    len(previous_registry_df) + len(registry_records)
)


# Define registry integrity verification

def verify_updated_registry(candidate_df):

    assert candidate_df['run_id'].is_unique

    assert len(candidate_df) == len(updated_registry_df)

    preserved_records = candidate_df.loc[
        ~candidate_df['run_id'].isin(svm_run_ids)
    ]

    pd.testing.assert_frame_equal(
        previous_registry_df.reset_index(drop=True),
        preserved_records.reset_index(drop=True),
        check_exact=True
    )

    for run_id, result in svm_results.items():

        rows = candidate_df.loc[
            candidate_df['run_id'] == run_id
        ]

        assert len(rows) == 1

        saved_record = rows.iloc[0]

        assert saved_record['status'] == 'Completed'

        assert saved_record['model'] == MODEL_NAME

        assert saved_record['feature_representation'] == (
            result['spec']['representation']
        )

        assert np.isclose(
            float(saved_record['validation_macro_ap']),
            result['summary']['Macro AP'],
            rtol=0,
            atol=1e-12
        )

        assert np.isclose(
            float(saved_record['validation_macro_f1']),
            result['summary']['Optimized Macro F1'],
            rtol=0,
            atol=1e-12
        )

        assert np.isclose(
            float(saved_record['training_time_seconds']),
            result['summary']['Training Time (s)'],
            rtol=0,
            atol=1e-12
        )

        assert int(saved_record['training_samples']) == len(y_train)

        assert int(saved_record['seed']) == SEED

        assert (
            PROJECT_ROOT / saved_record['model_path']
        ).is_file()

        assert (
            PROJECT_ROOT / saved_record['config_path']
        ).is_file()

        assert (
            PROJECT_ROOT / saved_record['results_path']
        ).is_dir()


# Verify the temporary registry before replacing the original

temporary_registry_path = REGISTRY_PATH.with_suffix(
    '.tmp'
)

updated_registry_df.to_csv(
    temporary_registry_path,
    index=False
)

temporary_registry_df = pd.read_csv(
    temporary_registry_path,
    dtype=str,
    keep_default_na=False
)

verify_updated_registry(temporary_registry_df)

temporary_registry_path.replace(REGISTRY_PATH)


# Reload and verify the saved shared registry

saved_registry_df = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

verify_updated_registry(saved_registry_df)


# Display the updated experiment registry

registry_display_columns = [
    'run_id',
    'model',
    'feature_representation',
    'training_samples',
    'validation_macro_ap',
    'validation_macro_f1',
    'training_time_seconds',
    'status'
]

registry_display_df = saved_registry_df[
    registry_display_columns
].copy()

numeric_columns = [
    'training_samples',
    'validation_macro_ap',
    'validation_macro_f1',
    'training_time_seconds'
]

for column in numeric_columns:

    registry_display_df[column] = pd.to_numeric(
        registry_display_df[column]
    )

print("\nUpdated shared experiment registry")

display(registry_display_df.round(6))


# Generate the current validation leaderboard

leaderboard_df = (
    registry_display_df
    .loc[registry_display_df['status'] == 'Completed']
    .sort_values(
        ['validation_macro_ap', 'validation_macro_f1', 'run_id'],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

leaderboard_df.insert(
    0,
    'Rank',
    np.arange(1, len(leaderboard_df) + 1)
)

print("\nCurrent validation leaderboard")

display(leaderboard_df.round(6))

leaderboard_df.to_csv(
    TABLES_DIR / 'svm_current_validation_leaderboard.csv',
    index=False
)


# Summarize the experiment registration

print("\nExperiment registration summary")

print("Previously registered experiments:",
      len(previous_registry_df))

print("Linear SVM experiments registered:",
      len(registry_records))

print("Total registered experiments:",
      len(saved_registry_df))

print("Duplicate run IDs:",
      saved_registry_df['run_id'].duplicated().sum())

print("Critical SVM artifact fingerprints verified:",
      registration_verification_df[
          'Fingerprinted Artifacts Verified'
      ].sum())

print("Previous experiment records: Preserved")

print("Selected Linear SVM run:", best_svm_run_id)

print("Current leading registered run:",
      leaderboard_df.loc[0, 'run_id'])

print("Registry verification: Passed")
print("Official test dataset: Not used")

print("\nLinear SVM experiments registered successfully.")

In [ ]:
# Verify notebook-level analysis artifacts

required_tables = [
    'environment_versions.csv',
    'environment_compatibility.csv',
    'previous_run_consistency.csv',
    'signed_margin_evaluation_verification.csv',
    'dataset_verification.csv',
    'feature_representation_verification.csv',
    'label_support_and_imbalance.csv',
    'svm_experiment_plan.csv',
    'svm_hyperparameter_rationale.csv',
    'svm_expected_balanced_class_weights.csv',
    'svm_word_baseline_model_comparison.csv',
    'svm_word_baseline_per_label_comparison.csv',
    'svm_word_vs_character_comparison.csv',
    'svm_word_vs_character_per_label.csv',
    'lr_vs_svm_character_comparison.csv',
    'lr_vs_svm_character_per_label.csv',
    'svm_representation_comparison.csv',
    'svm_representation_per_label_metrics.csv',
    'svm_representation_ap_comparison.csv',
    'svm_representation_f1_comparison.csv',
    'svm_combined_representation_effect.csv',
    'lr_vs_svm_combined_comparison.csv',
    'lr_vs_svm_combined_per_label.csv',
    'svm_representation_ranking.csv',
    'svm_representation_pr_curve_data.csv',
    'lr_vs_svm_all_representations.csv',
    'svm_representation_efficiency.csv',
    'svm_class_weighting_comparison.csv',
    'svm_class_weighting_per_label.csv',
    'svm_class_weighting_error_comparison.csv',
    'svm_class_weighting_threshold_comparison.csv',
    'svm_class_weighting_rare_labels.csv',
    'svm_weighting_threshold_interaction.csv',
    'lr_vs_svm_class_weighting_effect.csv',
    'lr_vs_svm_balanced_per_label.csv',
    'svm_all_experiments_comparison.csv',
    'svm_final_experiment_ranking.csv',
    'selected_lr_vs_svm_comparison.csv',
    'svm_registration_verification.csv',
    'svm_current_validation_leaderboard.csv'
]

required_figures = [
    'svm_representation_overall_comparison.png',
    'svm_representation_per_label_comparison.png',
    'svm_representation_precision_recall_curves.png',
    'lr_vs_svm_matched_representation_comparison.png',
    'svm_class_weighting_per_label_comparison.png',
    'lr_vs_svm_class_weighting_effect.png',
    'svm_final_configuration_comparison.png'
]

required_configs = [
    'svm_experiment_config.json',
    'svm_representation_selection.json',
    'svm_class_weighting_decision.json',
    'svm_final_selection.json'
]

verification_records = []

def verify_artifact(path, category, run_id='Shared'):

    available = path.is_file() and path.stat().st_size > 0

    verification_records.append({
        'Run ID': run_id,
        'Type': category,
        'Filename': path.name,
        'Path': str(path.relative_to(PROJECT_ROOT)),
        'Size Bytes': path.stat().st_size if available else 0,
        'Status': 'Available' if available else 'Missing'
    })


for filename in required_tables:
    verify_artifact(TABLES_DIR / filename, 'Analysis Table')

for filename in required_figures:
    verify_artifact(FIGURES_DIR / filename, 'Figure')

for filename in required_configs:
    verify_artifact(RUN_CONFIGS_DIR / filename, 'Configuration')


# Verify individual experiment artifacts

run_required_tables = [
    'feature_verification.csv',
    'classifier_training_summary.csv',
    'decision_score_distribution.csv',
    'default_overall_metrics.csv',
    'optimized_overall_metrics.csv',
    'default_per_label_metrics.csv',
    'optimized_per_label_metrics.csv',
    'optimized_thresholds.csv',
    'overall_threshold_comparison.csv',
    'per_label_threshold_comparison.csv',
    'run_summary.csv',
    'artifact_manifest.csv'
]

assert len(svm_results) == 4

verified_classifier_count = 0
verified_fingerprint_count = 0

for run_id, result in svm_results.items():

    run_dir = result['results_path']

    for filename in run_required_tables:

        verify_artifact(
            run_dir / 'tables' / filename,
            'Run Table',
            run_id
        )

    verify_artifact(
        result['model_path'],
        'Model',
        run_id
    )

    verify_artifact(
        result['predictions_path'],
        'Prediction',
        run_id
    )

    verify_artifact(
        RUN_CONFIGS_DIR / f'{run_id}.json',
        'Configuration',
        run_id
    )

    verify_artifact(
        RUN_CONFIGS_DIR / f'{run_id}_thresholds.json',
        'Configuration',
        run_id
    )

    for filename in [
        'training_log.json',
        'training_warnings.json'
    ]:

        verify_artifact(
            run_dir / 'logs' / filename,
            'Training Log',
            run_id
        )

    # Verify classifier convergence

    classifier_df = pd.read_csv(
        run_dir / 'tables' / 'classifier_training_summary.csv'
    )

    assert classifier_df['Label'].tolist() == LABELS
    assert classifier_df['Convergence'].eq('Passed').all()

    assert (
        classifier_df['Iterations']
        < classifier_df['Maximum Iterations']
    ).all()

    verified_classifier_count += len(classifier_df)

    # Recheck critical artifact fingerprints

    manifest_df = pd.read_csv(
        run_dir / 'tables' / 'artifact_manifest.csv'
    )

    assert len(manifest_df) == 7
    assert manifest_df['Run ID'].eq(run_id).all()

    for _, artifact in manifest_df.iterrows():

        path = PROJECT_ROOT / artifact['Path']

        assert path.is_file()
        assert path.stat().st_size == artifact['Size Bytes']
        assert calculate_sha256(path) == artifact['SHA256']

        verified_fingerprint_count += 1

    # Verify saved training and reconstruction status

    with open(
        run_dir / 'logs' / 'training_log.json',
        'r'
    ) as file:

        training_log = json.load(file)

    assert training_log['status'] == 'Completed'
    assert training_log['convergence_warnings'] == 0
    assert training_log['all_classifiers_converged']
    assert training_log['model_reconstruction_verified']
    assert training_log['metric_reconstruction_verified']
    assert training_log['score_alignment_verified']
    assert training_log['threshold_roundtrip_verified']
    assert training_log['prediction_roundtrip_verified']
    assert not training_log['official_test_used']


# Verify shared vectorizer dependencies

vectorizer_records = feature_manifest.loc[
    feature_manifest['Artifact Type'] == 'Vectorizer'
]

assert len(vectorizer_records) == 4

for _, record in vectorizer_records.iterrows():

    path = PROJECT_ROOT / record['Path']

    assert path.is_file()
    assert calculate_sha256(path) == record['SHA256']

    verify_artifact(path, 'Shared Dependency')

verify_artifact(
    SRC_DIR / 'evaluation_utils.py',
    'Shared Dependency'
)

verify_artifact(
    FEATURES_DIR / 'feature_artifact_manifest.csv',
    'Shared Dependency'
)

assert calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
) == evaluation_hash

assert calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
) == feature_manifest_hash


# Consolidate artifact verification results

final_verification_df = pd.DataFrame(
    verification_records
)

assert final_verification_df['Path'].is_unique

final_verification_df.to_csv(
    TABLES_DIR / 'svm_final_artifact_verification.csv',
    index=False
)

missing_artifacts_df = final_verification_df.loc[
    final_verification_df['Status'] == 'Missing'
]

if not missing_artifacts_df.empty:
    display(missing_artifacts_df)

assert missing_artifacts_df.empty

print("Final artifact verification")

display(
    final_verification_df
    .groupby(['Type', 'Status'])
    .size()
    .reset_index(name='Artifacts')
)

assert verified_classifier_count == 24
assert verified_fingerprint_count == 28


# Verify final SVM selection

with open(SVM_FINAL_SELECTION_PATH, 'r') as file:
    saved_selection = json.load(file)

assert saved_selection['selected_run_id'] == best_svm_run_id

assert saved_selection['selected_representation'] == (
    best_svm_result['spec']['representation']
)

assert saved_selection['selected_class_weight'] == (
    best_svm_result['spec']['class_weight']
)

assert saved_selection['score_type'] == SCORE_TYPE
assert not saved_selection['final_project_model_frozen']
assert not saved_selection['official_test_used']

assert np.isclose(
    saved_selection['validation_macro_ap'],
    best_svm_summary['Macro AP'],
    rtol=0,
    atol=1e-12
)

assert best_svm_summary['Macro AP'] == (
    svm_comparison_df['Macro AP'].max()
)

for field in [
    'model_path',
    'threshold_path',
    'predictions_path'
]:
    assert (PROJECT_ROOT / saved_selection[field]).is_file()


# Verify registration of all completed experiments

saved_registry_df = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert saved_registry_df['run_id'].is_unique

assert set(svm_results).issubset(
    set(saved_registry_df['run_id'])
)

assert set(required_previous_runs).issubset(
    set(saved_registry_df['run_id'])
)

for run_id, result in svm_results.items():

    record = saved_registry_df.loc[
        saved_registry_df['run_id'] == run_id
    ].iloc[0]

    assert record['status'] == 'Completed'

    assert np.isclose(
        float(record['validation_macro_ap']),
        result['summary']['Macro AP'],
        rtol=0,
        atol=1e-12
    )

    assert np.isclose(
        float(record['validation_macro_f1']),
        result['summary']['Optimized Macro F1'],
        rtol=0,
        atol=1e-12
    )


# Record evidence-based methodological decisions

DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

decisions_df = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

assert decisions_df['decision_id'].is_unique

svm_decisions = [
    {
        'decision_id': 'D033',
        'component': 'Linear SVM configuration',
        'decision': (
            'Use LinearSVC with L2 regularization, '
            'squared hinge loss, primal optimization and C=1.0'
        ),
        'justification': (
            'Suitable baseline for high-dimensional sparse '
            'text classification. All 24 binary classifiers '
            'converged without warnings. The configuration '
            'was held constant across controlled comparisons.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D034',
        'component': 'Linear SVM score evaluation',
        'decision': (
            'Evaluate signed decision margins using a default '
            'threshold of zero and per-label validation thresholds'
        ),
        'justification': (
            'LinearSVC produces signed margins rather than '
            'probabilities. Negative thresholds are supported '
            'by the verified shared evaluation module. '
            'Pooled micro ranking metrics require caution '
            'because independent classifiers may have '
            'different score scales.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D035',
        'component': 'Linear SVM representation selection',
        'decision': (
            f'Select {best_representation} for the '
            'balanced class weighting experiment'
        ),
        'justification': (
            f"Combined TF-IDF achieves Macro AP of "
            f"{combined_result['summary']['Macro AP']:.6f}, "
            f"compared with "
            f"{char_result['summary']['Macro AP']:.6f} for "
            f"character and "
            f"{word_result['summary']['Macro AP']:.6f} for word. "
            'The comparison uses a fixed total feature budget. '
            'Statistical significance has not been established.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D036',
        'component': 'Linear SVM class imbalance handling',
        'decision': (
            'Retain balanced weighting as a controlled '
            'extension and retain its negative result'
        ),
        'justification': (
            f"Balanced weighting changes Macro AP from "
            f"{unweighted_summary['Macro AP']:.6f} to "
            f"{weighted_summary['Macro AP']:.6f} and "
            f"optimized Macro F1 from "
            f"{unweighted_summary['Optimized Macro F1']:.6f} to "
            f"{weighted_summary['Optimized Macro F1']:.6f}. "
            'Although default Macro F1 and severe toxicity '
            'performance improve, overall selection metrics '
            'decline after threshold optimization. '
            'This contrasts with Logistic Regression.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D037',
        'component': 'Linear SVM model selection',
        'decision': f'Select {best_svm_run_id}',
        'justification': (
            f"Highest validation Macro AP of "
            f"{best_svm_summary['Macro AP']:.6f} "
            'among the four evaluated SVM configurations. '
            'Selection is restricted to Linear SVM and '
            'does not freeze the final project model.'
        ),
        'status': 'Selected'
    },
    {
        'decision_id': 'D038',
        'component': 'Linear SVM experiment scope',
        'decision': (
            'Retain four controlled Linear SVM experiments '
            'without additional regularization or weight searches'
        ),
        'justification': (
            'Representation, class weighting and threshold '
            'effects have been evaluated. Additional tuning '
            'is deprioritized to preserve resources for '
            'remaining classical models and required '
            'transformer experiments. Conclusions remain '
            'conditional on the evaluated configurations.'
        ),
        'status': 'Implemented'
    }
]

assert set(decisions_df.columns) == set(
    svm_decisions[0].keys()
)

for record in svm_decisions:

    mask = decisions_df['decision_id'].eq(
        record['decision_id']
    )

    if mask.any():

        assert decisions_df.loc[
            mask, 'component'
        ].eq(record['component']).all()

        for column, value in record.items():
            decisions_df.loc[mask, column] = value

    else:

        decisions_df = pd.concat(
            [decisions_df, pd.DataFrame([record])],
            ignore_index=True
        )

assert decisions_df['decision_id'].is_unique

temporary_decisions_path = DECISION_LOG_PATH.with_suffix(
    '.tmp'
)

decisions_df.to_csv(
    temporary_decisions_path,
    index=False
)

saved_decisions_df = pd.read_csv(
    temporary_decisions_path,
    dtype=str,
    keep_default_na=False
)

pd.testing.assert_frame_equal(
    decisions_df.reset_index(drop=True),
    saved_decisions_df.reset_index(drop=True),
    check_exact=True
)

temporary_decisions_path.replace(DECISION_LOG_PATH)

print("\nNew methodological decisions")

display(
    decisions_df.loc[
        decisions_df['decision_id'].isin([
            record['decision_id']
            for record in svm_decisions
        ])
    ][['decision_id', 'component', 'decision', 'status']]
)


# Construct the final experiment results table

summary_table = [
    '| Configuration | Macro AP | Macro ROC-AUC | Default Macro F1 | Optimized Macro F1 | Optimized Micro F1 |',
    '|---|---:|---:|---:|---:|---:|'
]

for _, row in svm_final_ranking_df.iterrows():

    summary_table.append(
        f"| {run_display_names[row['Run ID']]} "
        f"| {row['Macro AP']:.6f} "
        f"| {row['Macro ROC-AUC']:.6f} "
        f"| {row['Default Macro F1']:.6f} "
        f"| {row['Optimized Macro F1']:.6f} "
        f"| {row['Optimized Micro F1']:.6f} |"
    )

summary_table_text = '\n'.join(summary_table)


# Save the final research summary

summary = f"""# Notebook 03.03 - Linear SVM

## Objective

Evaluate margin-based multi-label classification using
word, character and combined TF-IDF representations.
Compare Linear SVM with Naive Bayes and Logistic Regression,
investigate balanced class weighting and optimize
individual classification thresholds.

## Experimental Configuration

- Training samples: {len(y_train):,}
- Validation samples: {len(y_val):,}
- Features per representation: {FEATURE_BUDGET:,}
- Estimator: LinearSVC
- Classification strategy: One-vs-Rest
- Loss: Squared hinge
- Regularization: L2
- C: 1.0
- Optimization: Primal
- Random seed: {SEED}
- Default decision threshold: 0.0
- Completed experiments: {len(svm_results)}

## Validation Results

{summary_table_text}

## Research Findings

### Common word-feature comparison

Linear SVM achieves higher Macro AP and optimized Macro F1
than Naive Bayes and Logistic Regression using the same
word TF-IDF representation and datasets.

Macro AP:
- Naive Bayes: {nb_macro_ap:.6f}
- Logistic Regression: {lr_macro_ap:.6f}
- Linear SVM: {word_result['summary']['Macro AP']:.6f}

Logistic Regression retains higher Macro ROC-AUC than
Linear SVM in the matched word-feature comparison.

### Feature representation

Character features improve AP and optimized F1 for all
six labels relative to word features.

Combined TF-IDF achieves the highest unweighted Macro AP
under the fixed total feature budget.

Character-only features remain stronger for identity hate
than the combined representation.

### Matched Logistic Regression comparison

Linear SVM achieves higher Macro AP across word, character
and combined unweighted representations.

Logistic Regression consistently achieves higher Macro ROC-AUC.

These results are conditional on the evaluated hyperparameters
and the common validation split.

### Class weighting

Balanced class weighting changes SVM Macro AP from
{unweighted_summary['Macro AP']:.6f}
to {weighted_summary['Macro AP']:.6f}.

Default Macro F1 increases from
{unweighted_summary['Default Macro F1']:.6f}
to {weighted_summary['Default Macro F1']:.6f}.

However, optimized Macro F1 decreases from
{unweighted_summary['Optimized Macro F1']:.6f}
to {weighted_summary['Optimized Macro F1']:.6f}.

Rare-label mean AP changes from
{rare_label_comparison_df['Unweighted AP'].mean():.6f}
to {rare_label_comparison_df['Balanced AP'].mean():.6f}.

Rare-label mean F1 changes from
{rare_label_comparison_df['Unweighted F1'].mean():.6f}
to {rare_label_comparison_df['Balanced F1'].mean():.6f}.

Severe toxicity is the only label with improvements
in both AP and optimized F1 after weighting.

The weighted model requires
{weighting_effect['training_time_ratio']:.4f}
times the unweighted combined model's training time
in the recorded single runs.

The weighted configuration is retained as experimental
evidence rather than selected as the strongest SVM model.

### Algorithm-dependent imbalance handling

Balanced weighting improves Logistic Regression Macro AP
by {lr_weighted_summary['Macro AP'] - lr_combined_summary['Macro AP']:.6f}.

The same balanced weighting strategy changes Linear SVM
Macro AP by {macro_ap_change:.6f}.

This demonstrates that imbalance handling effectiveness
depends on the evaluated learning algorithm and configuration.

### Threshold optimization

Per-label validation threshold optimization improves
Macro F1 across all four SVM configurations.

Threshold optimization leaves ranking metrics unchanged.

Unweighted SVM requires negative optimized thresholds
for all six labels in the evaluated representations.

Balanced SVM produces a mixture of positive and negative
optimized thresholds.

## Selected Linear SVM Configuration

- Run ID: {best_svm_run_id}
- Representation: {best_svm_result['spec']['representation']}
- Class weighting: {best_svm_result['spec']['class_weight']}
- Validation Macro AP: {best_svm_summary['Macro AP']:.6f}
- Validation Macro ROC-AUC: {best_svm_summary['Macro ROC-AUC']:.6f}
- Optimized Macro F1: {best_svm_summary['Optimized Macro F1']:.6f}
- Optimized Micro F1: {best_svm_summary['Optimized Micro F1']:.6f}
- Training time: {best_svm_summary['Training Time (s)']:.6f} seconds

## Leading Linear Model

Balanced combined-feature Logistic Regression remains
the leading evaluated linear classifier by validation
Macro AP of {lr_final_selection['validation_macro_ap']:.6f}.

The selected SVM configuration achieves higher optimized
Micro F1 and stronger threat AP and optimized F1.

## Reproducibility

- Four completed SVM experiments.
- All {verified_classifier_count} binary classifiers converged.
- All saved models reproduced their validation scores.
- Decision score alignment verified.
- Full-precision thresholds preserved.
- Prediction and metric reconstruction verified.
- {verified_fingerprint_count} critical artifact fingerprint checks passed.
- All four SVM experiments registered.
- Previous experiment records preserved.

## Limitations

- Results are based on a single validation split.
- Optimized validation F1 may be optimistic.
- Statistical significance has not been established.
- Signed SVM margins are not calibrated probabilities.
- Pooled micro ranking metrics may be affected by
  differences in independent classifier score scales.
- Regularization strength and alternative class weights
  were not individually optimized.
- The fixed feature budget does not establish the optimal
  vocabulary size or allocation.
- Training times are single-run observations.
- Sparse matrix memory does not represent peak runtime RAM.
- The observed negative weighting result does not establish
  that class weighting is generally unsuitable for SVM.
- The shared evaluation module is verified for multi-label
  inputs; its single-label limitation remains unchanged.

## Evaluation Protocol

Macro Average Precision remains the primary model
selection criterion.

Official test data has not been used for model fitting,
threshold optimization or model selection.

The final project model has not been frozen.

## Next Stage

Notebook 03.04 - Decision Tree.
"""

summary_path = RESULTS_DIR / 'summary.md'

summary_path.write_text(
    summary,
    encoding='utf-8'
)

assert summary_path.is_file()
assert summary_path.stat().st_size > 0


# Summarize notebook completion

print("\nLinear SVM completion summary")

print("Notebook-level analysis tables:", len(required_tables))
print("Individual experiment tables:",
      len(run_required_tables) * len(svm_results))
print("Research figures:", len(required_figures))
print("Completed SVM experiments:", len(svm_results))
print("Verified binary classifiers:", verified_classifier_count)
print("Critical fingerprint checks:", verified_fingerprint_count)
print("Total required artifact entries:", len(final_verification_df))
print("Missing artifacts:", len(missing_artifacts_df))
print("Total methodological decisions:", len(decisions_df))
print("Total registered experiments:", len(saved_registry_df))
print("Selected SVM configuration:", best_svm_run_id)
print("Leading linear configuration:",
      lr_final_selection['selected_run_id'])
print("Notebook summary saved.")
print("Official test dataset: Not used")
print("Final project model frozen: No")

print("\nLinear SVM notebook completed successfully.")